In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df.columns.tolist())

In [ ]:
al_data = df[["author", "year", "number", "al_event", "al_rate"]].copy()

al_data

In [ ]:
al_data["calculated_rate"] = (
    al_data["al_event"] / al_data["number"] * 100
)

al_data[["author", "number", "al_event", "al_rate", "calculated_rate"]]

al_data.loc[18, "al_rate"] = 25.0

In [ ]:
al_data["author"] = al_data["author"].ffill()
al_data["year"] = al_data["year"].ffill()

al_data

In [ ]:
df = pd.read_excel("ICG2.xlsx", header=1)

df.head()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df.columns.tolist())


al_data = df[["study_id", "author", "year", "number", "al_event", "al_rate"]].copy()

al_data


print(al_data[["number", "al_event"]].dtypes)

print("\nMissing values:")
print(al_data[["number", "al_event"]].isna().sum())

print("\nNumber of rows:", len(al_data))

In [ ]:
al_data["proportion"] = al_data["al_event"] / al_data["number"]

al_data[["study_id", "number", "al_event", "proportion"]]

In [ ]:
al_data["logit_p"] = np.log(
    al_data["proportion"] / (1 - al_data["proportion"])
)

al_data[["study_id", "proportion", "logit_p"]]

In [ ]:
al_data["variance"] = (
    1 / al_data["al_event"] +
    1 / (al_data["number"] - al_data["al_event"])
)

al_data["se"] = np.sqrt(al_data["variance"])

al_data[
    ["study_id", "number", "al_event", "proportion",
     "logit_p", "variance", "se"]
]

In [ ]:
# Fixed-effect weights
al_data["w_fixed"] = 1 / al_data["variance"]

# Fixed-effect pooled logit
fixed_logit = (
    (al_data["w_fixed"] * al_data["logit_p"]).sum()
    / al_data["w_fixed"].sum()
)

# Cochran's Q
Q = (
    al_data["w_fixed"]
    * (al_data["logit_p"] - fixed_logit) ** 2
).sum()

k = len(al_data)
df_Q = k - 1

# I-squared
I2 = max(0, (Q - df_Q) / Q) * 100

# DerSimonian-Laird tau-squared
C = (
    al_data["w_fixed"].sum()
    - (al_data["w_fixed"] ** 2).sum()
      / al_data["w_fixed"].sum()
)

tau2 = max(0, (Q - df_Q) / C)

print("Number of cohorts:", k)
print("Q =", round(Q, 3))
print("df =", df_Q)
print("I² =", round(I2, 1), "%")
print("tau² =", round(tau2, 4))

In [ ]:
# Random-effects weights
al_data["w_random"] = 1 / (al_data["variance"] + tau2)

# Pooled logit proportion
pooled_logit = (
    (al_data["w_random"] * al_data["logit_p"]).sum()
    / al_data["w_random"].sum()
)

# Standard error of pooled logit
se_pooled = np.sqrt(1 / al_data["w_random"].sum())

# 95% CI on logit scale
lower_logit = pooled_logit - 1.96 * se_pooled
upper_logit = pooled_logit + 1.96 * se_pooled

# Back-transform logit to proportion
def inv_logit(x):
    return np.exp(x) / (1 + np.exp(x))

pooled_prop = inv_logit(pooled_logit)
lower_prop = inv_logit(lower_logit)
upper_prop = inv_logit(upper_logit)

print("Pooled AL rate =", round(pooled_prop * 100, 2), "%")
print(
    "95% CI =",
    round(lower_prop * 100, 2), "% to",
    round(upper_prop * 100, 2), "%"
)

In [ ]:
from scipy.optimize import minimize_scalar

# Effect sizes and within-study variances
yi = al_data["logit_p"].to_numpy()
vi = al_data["variance"].to_numpy()
k = len(yi)

# Restricted log-likelihood
def reml_objective(tau2):
    wi = 1 / (vi + tau2)
    mu = np.sum(wi * yi) / np.sum(wi)

    value = (
        np.sum(np.log(vi + tau2))
        + np.log(np.sum(wi))
        + np.sum(wi * (yi - mu)**2)
    )

    return value

# Estimate tau² using REML
result = minimize_scalar(
    reml_objective,
    bounds=(0, 10),
    method="bounded"
)

tau2_reml = result.x
tau_reml = np.sqrt(tau2_reml)

print("REML tau² =", round(tau2_reml, 4))
print("REML tau  =", round(tau_reml, 4))
print("Converged =", result.success)

In [ ]:
# Random-effects weights using REML tau²
al_data["w_reml"] = 1 / (al_data["variance"] + tau2_reml)

# Pooled effect on logit scale
pooled_logit_reml = (
    np.sum(al_data["w_reml"] * al_data["logit_p"])
    / np.sum(al_data["w_reml"])
)

# Standard error
se_reml = np.sqrt(1 / np.sum(al_data["w_reml"]))

# Conventional 95% CI (normal/Wald)
lower_logit_reml = pooled_logit_reml - 1.96 * se_reml
upper_logit_reml = pooled_logit_reml + 1.96 * se_reml

# Back-transform to proportions
pooled_prop_reml = inv_logit(pooled_logit_reml)
lower_prop_reml = inv_logit(lower_logit_reml)
upper_prop_reml = inv_logit(upper_logit_reml)

print("REML pooled AL rate =", round(pooled_prop_reml * 100, 2), "%")
print(
    "95% CI =",
    round(lower_prop_reml * 100, 2), "% to",
    round(upper_prop_reml * 100, 2), "%"
)
print("Pooled logit =", round(pooled_logit_reml, 4))
print("SE =", round(se_reml, 4))

In [ ]:
from scipy.stats import t

# Number of cohorts
k = len(al_data)

# REML weights
w = al_data["w_reml"].to_numpy()
y = al_data["logit_p"].to_numpy()

# Hartung-Knapp variance adjustment
q_hk = (
    np.sum(w * (y - pooled_logit_reml)**2)
    / (k - 1)
)

var_hk = q_hk / np.sum(w)
se_hk = np.sqrt(var_hk)

# t critical value with k-1 degrees of freedom
t_crit = t.ppf(0.975, df=k-1)

# HK confidence interval on logit scale
lower_hk_logit = pooled_logit_reml - t_crit * se_hk
upper_hk_logit = pooled_logit_reml + t_crit * se_hk

# Back-transform
lower_hk = inv_logit(lower_hk_logit)
upper_hk = inv_logit(upper_hk_logit)

print("Pooled AL rate =", round(pooled_prop_reml * 100, 2), "%")
print(
    "Hartung-Knapp 95% CI =",
    round(lower_hk * 100, 2), "% to",
    round(upper_hk * 100, 2), "%"
)

print("HK SE =", round(se_hk, 4))
print("t critical =", round(t_crit, 4))
print("q =", round(q_hk, 4))

In [ ]:
# 95% prediction interval using REML heterogeneity

# Prediction standard error
se_prediction = np.sqrt(tau2_reml + se_reml**2)

# t critical value
t_pred = t.ppf(0.975, df=k-2)

# Prediction interval on logit scale
lower_pi_logit = pooled_logit_reml - t_pred * se_prediction
upper_pi_logit = pooled_logit_reml + t_pred * se_prediction

# Back-transform to proportions
lower_pi = inv_logit(lower_pi_logit)
upper_pi = inv_logit(upper_pi_logit)

print("Pooled AL rate =", round(pooled_prop_reml * 100, 2), "%")
print(
    "95% prediction interval =",
    round(lower_pi * 100, 2), "% to",
    round(upper_pi * 100, 2), "%"
)

In [ ]:
# Sensitivity analysis:
# exclude the two small case series

sensitivity = al_data[
    ~al_data["study_id"].isin([
        "Bongiolatti S, 2016",
        "Fortich S, 2026"
    ])
].copy()

print("Original cohorts:", len(al_data))
print("Sensitivity-analysis cohorts:", len(sensitivity))

sensitivity[
    ["study_id", "number", "al_event", "proportion"]
]

In [ ]:
# Data for sensitivity analysis
yi_sens = sensitivity["logit_p"].to_numpy()
vi_sens = sensitivity["variance"].to_numpy()
k_sens = len(sensitivity)

# REML objective function
def reml_objective_sens(tau2):
    w = 1 / (vi_sens + tau2)
    mu = np.sum(w * yi_sens) / np.sum(w)

    return (
        np.sum(np.log(vi_sens + tau2))
        + np.log(np.sum(w))
        + np.sum(w * (yi_sens - mu)**2)
    )

# Estimate tau² with REML
result_sens = minimize_scalar(
    reml_objective_sens,
    bounds=(0, 10),
    method="bounded"
)

tau2_sens = result_sens.x

# Random-effects weights
w_sens = 1 / (vi_sens + tau2_sens)

# Pooled logit
pooled_logit_sens = (
    np.sum(w_sens * yi_sens) /
    np.sum(w_sens)
)

# Standard error
se_sens = np.sqrt(1 / np.sum(w_sens))

# 95% CI
lower_sens_logit = pooled_logit_sens - 1.96 * se_sens
upper_sens_logit = pooled_logit_sens + 1.96 * se_sens

# Back-transform
pooled_sens = inv_logit(pooled_logit_sens)
lower_sens = inv_logit(lower_sens_logit)
upper_sens = inv_logit(upper_sens_logit)

print("Number of cohorts =", k_sens)
print("REML tau² =", round(tau2_sens, 4))
print("Pooled AL =", round(pooled_sens * 100, 2), "%")
print(
    "95% CI =",
    round(lower_sens * 100, 2), "% to",
    round(upper_sens * 100, 2), "%"
)

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# Calculate Wilson 95% CI for each cohort
ci = [
    proportion_confint(
        count=x,
        nobs=n,
        alpha=0.05,
        method="wilson"
    )
    for x, n in zip(al_data["al_event"], al_data["number"])
]

al_data["ci_lower"] = [x[0] for x in ci]
al_data["ci_upper"] = [x[1] for x in ci]

# Random-effects weights as percentages
al_data["weight_percent"] = (
    al_data["w_reml"] /
    al_data["w_reml"].sum()
    * 100
)

al_data[
    [
        "study_id",
        "al_event",
        "number",
        "proportion",
        "ci_lower",
        "ci_upper",
        "weight_percent"
    ]
]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Data for forest plot
# -----------------------------
plot_data = al_data.copy()

k = len(plot_data)
y = np.arange(k, 0, -1)

# Convert proportions to %
effect = plot_data["proportion"] * 100
lower = plot_data["ci_lower"] * 100
upper = plot_data["ci_upper"] * 100

# -----------------------------
# Figure
# -----------------------------
fig, ax = plt.subplots(figsize=(13, 10))

# Confidence intervals
for i in range(k):
    ax.plot(
        [lower.iloc[i], upper.iloc[i]],
        [y[i], y[i]],
        linewidth=1.3
    )

# Study estimates
# Square size proportional to random-effects weight
sizes = 25 + plot_data["weight_percent"] * 12

ax.scatter(
    effect,
    y,
    s=sizes,
    marker="s",
    zorder=3
)

# -----------------------------
# Pooled estimate
# -----------------------------
pooled = 10.02
pooled_low = 7.24
pooled_high = 13.70

diamond_y = 0

diamond_x = [
    pooled_low,
    pooled,
    pooled_high,
    pooled
]

diamond_y_values = [
    diamond_y,
    diamond_y + 0.28,
    diamond_y,
    diamond_y - 0.28
]

ax.fill(
    diamond_x,
    diamond_y_values,
    alpha=0.7
)

# -----------------------------
# Prediction interval
# -----------------------------
pred_low = 3.10
pred_high = 27.92

ax.plot(
    [pred_low, pred_high],
    [-1, -1],
    linewidth=2
)

ax.plot(
    [pred_low, pred_low],
    [-1.15, -0.85],
    linewidth=2
)

ax.plot(
    [pred_high, pred_high],
    [-1.15, -0.85],
    linewidth=2
)

ax.scatter(
    pooled,
    -1,
    marker="D",
    s=35,
    zorder=3
)

# -----------------------------
# Left-hand study labels
# -----------------------------
for i in range(k):
    ax.text(
        -2,
        y[i],
        plot_data["study_id"].iloc[i],
        ha="right",
        va="center",
        fontsize=9
    )

# Events / total
for i in range(k):
    ax.text(
        31,
        y[i],
        f'{int(plot_data["al_event"].iloc[i])}/{int(plot_data["number"].iloc[i])}',
        ha="center",
        va="center",
        fontsize=9
    )

# Effect + CI
for i in range(k):
    ax.text(
        42,
        y[i],
        f'{effect.iloc[i]:.1f}% '
        f'[{lower.iloc[i]:.1f}–{upper.iloc[i]:.1f}]',
        ha="center",
        va="center",
        fontsize=9
    )

# Weight
for i in range(k):
    ax.text(
        55,
        y[i],
        f'{plot_data["weight_percent"].iloc[i]:.1f}%',
        ha="center",
        va="center",
        fontsize=9
    )

# -----------------------------
# Column headers
# -----------------------------
ax.text(-2, k + 1.2, "Study", ha="right", fontweight="bold")
ax.text(31, k + 1.2, "Events/Total", ha="center", fontweight="bold")
ax.text(42, k + 1.2, "AL rate [95% CI]", ha="center", fontweight="bold")
ax.text(55, k + 1.2, "Weight", ha="center", fontweight="bold")

# -----------------------------
# Summary labels
# -----------------------------
ax.text(
    -2,
    0,
    "Random-effects model",
    ha="right",
    va="center",
    fontweight="bold"
)

ax.text(
    42,
    0,
    f"{pooled:.2f}% [{pooled_low:.2f}–{pooled_high:.2f}]",
    ha="center",
    va="center",
    fontweight="bold"
)

ax.text(
    -2,
    -1,
    "Prediction interval",
    ha="right",
    va="center",
    fontsize=9
)

ax.text(
    42,
    -1,
    f"{pred_low:.2f}%–{pred_high:.2f}%",
    ha="center",
    va="center",
    fontsize=9
)

# -----------------------------
# Formatting
# -----------------------------
ax.set_xlim(-20, 62)
ax.set_ylim(-2.5, k + 2)

ax.set_xlabel("Anastomotic leak rate (%)", fontsize=11)

ax.set_yticks([])

# Reference line at pooled estimate
ax.axvline(
    pooled,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)

# Heterogeneity information
ax.text(
    -2,
    -2,
    "REML: τ² = 0.3165; I² = 62.0%",
    ha="right",
    va="center",
    fontsize=9
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# PUBLICATION-QUALITY FOREST PLOT
# ============================================================

plot_data = al_data.copy().reset_index(drop=True)

k = len(plot_data)
y = np.arange(k, 0, -1)

# Convert proportions to percentages
effect = plot_data["proportion"].to_numpy() * 100
lower = plot_data["ci_lower"].to_numpy() * 100
upper = plot_data["ci_upper"].to_numpy() * 100
weights = plot_data["weight_percent"].to_numpy()

# Main meta-analysis results
pooled = 10.02
pooled_low = 7.24
pooled_high = 13.70

pred_low = 3.10
pred_high = 27.92

I2 = 62.0
tau2 = 0.3165

# ============================================================
# Figure layout
# ============================================================

fig = plt.figure(figsize=(16, 10))

gs = GridSpec(
    1, 5,
    width_ratios=[3.2, 4.2, 1.4, 3.0, 1.2],
    wspace=0.03
)

ax_study  = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1])
ax_events = fig.add_subplot(gs[0, 2])
ax_effect = fig.add_subplot(gs[0, 3])
ax_weight = fig.add_subplot(gs[0, 4])

axes = [ax_study, ax_forest, ax_events, ax_effect, ax_weight]

# Common Y limits
for ax in axes:
    ax.set_ylim(-3.2, k + 2)

# ============================================================
# LEFT COLUMN — STUDY
# ============================================================

ax_study.set_xlim(0, 1)

ax_study.text(
    0.02, k + 1.2,
    "Study",
    fontsize=11,
    fontweight="bold",
    va="center"
)

for i in range(k):
    ax_study.text(
        0.02,
        y[i],
        str(plot_data.loc[i, "study_id"]),
        fontsize=9.5,
        va="center"
    )

ax_study.text(
    0.02,
    0,
    "Random-effects model",
    fontsize=10,
    fontweight="bold",
    va="center"
)

ax_study.text(
    0.02,
    -1,
    "Prediction interval",
    fontsize=9.5,
    va="center"
)

ax_study.text(
    0.02,
    -2.2,
    f"Heterogeneity: I² = {I2:.1f}%; τ² = {tau2:.4f}",
    fontsize=9,
    va="center"
)

# ============================================================
# FOREST PANEL
# ============================================================

# Choose scale wide enough for all study CIs
xmax = max(35, np.ceil(max(upper) / 5) * 5)

ax_forest.set_xlim(0, xmax)

# Study confidence intervals
for i in range(k):
    ax_forest.hlines(
        y[i],
        lower[i],
        upper[i],
        linewidth=1.3
    )

# Squares proportional to random-effects weight
square_sizes = 20 + weights * 14

ax_forest.scatter(
    effect,
    y,
    s=square_sizes,
    marker="s",
    zorder=3
)

# Vertical reference line = pooled estimate
ax_forest.axvline(
    pooled,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)

# Pooled diamond
diamond_x = [
    pooled_low,
    pooled,
    pooled_high,
    pooled
]

diamond_y = [
    0,
    0.30,
    0,
    -0.30
]

ax_forest.fill(
    diamond_x,
    diamond_y,
    alpha=0.8
)

# Prediction interval
ax_forest.hlines(
    -1,
    pred_low,
    pred_high,
    linewidth=2
)

ax_forest.vlines(
    [pred_low, pred_high],
    -1.15,
    -0.85,
    linewidth=2
)

ax_forest.scatter(
    pooled,
    -1,
    marker="D",
    s=35,
    zorder=3
)

ax_forest.set_xlabel(
    "Anastomotic leak rate (%)",
    fontsize=10
)

ax_forest.set_yticks([])

# ============================================================
# EVENTS / TOTAL
# ============================================================

ax_events.set_xlim(0, 1)

ax_events.text(
    0.5,
    k + 1.2,
    "Events/Total",
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center"
)

for i in range(k):
    ax_events.text(
        0.5,
        y[i],
        f'{int(plot_data.loc[i, "al_event"])}/{int(plot_data.loc[i, "number"])}',
        fontsize=9.5,
        ha="center",
        va="center"
    )

# ============================================================
# EFFECT + 95% CI
# ============================================================

ax_effect.set_xlim(0, 1)

ax_effect.text(
    0.5,
    k + 1.2,
    "AL rate [95% CI]",
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center"
)

for i in range(k):
    ax_effect.text(
        0.5,
        y[i],
        f"{effect[i]:.1f}% [{lower[i]:.1f}–{upper[i]:.1f}]",
        fontsize=9.5,
        ha="center",
        va="center"
    )

# Pooled result
ax_effect.text(
    0.5,
    0,
    f"{pooled:.2f}% [{pooled_low:.2f}–{pooled_high:.2f}]",
    fontsize=10,
    fontweight="bold",
    ha="center",
    va="center"
)

# Prediction interval
ax_effect.text(
    0.5,
    -1,
    f"{pred_low:.2f}%–{pred_high:.2f}%",
    fontsize=9.5,
    ha="center",
    va="center"
)

# ============================================================
# WEIGHT
# ============================================================

ax_weight.set_xlim(0, 1)

ax_weight.text(
    0.5,
    k + 1.2,
    "Weight",
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center"
)

for i in range(k):
    ax_weight.text(
        0.5,
        y[i],
        f"{weights[i]:.1f}%",
        fontsize=9.5,
        ha="center",
        va="center"
    )

ax_weight.text(
    0.5,
    0,
    "100%",
    fontsize=10,
    fontweight="bold",
    ha="center",
    va="center"
)

# ============================================================
# Formatting
# ============================================================

# Horizontal separator below headers
for ax in axes:
    ax.axhline(
        k + 0.6,
        linewidth=0.8,
        alpha=0.5
    )

# Remove unnecessary borders
for ax in [ax_study, ax_events, ax_effect, ax_weight]:
    ax.axis("off")

ax_forest.spines["top"].set_visible(False)
ax_forest.spines["right"].set_visible(False)
ax_forest.spines["left"].set_visible(False)

# Give enough room so first and last rows are not cropped
plt.subplots_adjust(
    top=0.96,
    bottom=0.10,
    left=0.04,
    right=0.98
)

# Save publication-quality versions
plt.savefig(
    "ICG_AL_forest_plot.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "ICG_AL_forest_plot.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Check robotic extent categories before subgroup analysis

subgroup_check = df[
    ["study_id", "number", "al_event", "robotic_extent"]
].copy()

subgroup_check

In [ ]:
df["robotic_extent"].value_counts(dropna=False)

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.stats import t
import numpy as np
import pandas as pd

# Prepare subgroup dataset
robot_data = df[
    df["robotic_extent"].isin(["Hybrid RAMIE", "Full RAMIE"])
][["study_id", "robotic_extent", "number", "al_event"]].copy()

# Calculate effect size
robot_data["p"] = robot_data["al_event"] / robot_data["number"]

robot_data["yi"] = np.log(
    robot_data["p"] / (1 - robot_data["p"])
)

robot_data["vi"] = (
    1 / robot_data["al_event"] +
    1 / (robot_data["number"] - robot_data["al_event"])
)


def subgroup_meta(data):

    yi = data["yi"].values
    vi = data["vi"].values
    k = len(data)

    # REML tau²
    def reml_negloglik(tau2):

        wi = 1 / (vi + tau2)
        mu = np.sum(wi * yi) / np.sum(wi)

        return 0.5 * (
            np.sum(np.log(vi + tau2))
            + np.log(np.sum(wi))
            + np.sum(wi * (yi - mu)**2)
        )

    result = minimize_scalar(
        reml_negloglik,
        bounds=(0, 10),
        method="bounded"
    )

    tau2 = result.x

    # Random-effects weights
    wi = 1 / (vi + tau2)

    pooled = np.sum(wi * yi) / np.sum(wi)

    # Hartung-Knapp variance
    q = np.sum(wi * (yi - pooled)**2) / (k - 1)

    se_hk = np.sqrt(q / np.sum(wi))

    tcrit = t.ppf(0.975, df=k-1)

    lower = pooled - tcrit * se_hk
    upper = pooled + tcrit * se_hk

    # Back-transform
    inv_logit = lambda x: np.exp(x) / (1 + np.exp(x))

    pooled_p = inv_logit(pooled)
    lower_p = inv_logit(lower)
    upper_p = inv_logit(upper)

    # I²
    w_fixed = 1 / vi
    fixed = np.sum(w_fixed * yi) / np.sum(w_fixed)
    Q = np.sum(w_fixed * (yi - fixed)**2)

    I2 = max(0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0

    return {
        "k": k,
        "pooled_AL": pooled_p * 100,
        "lower_CI": lower_p * 100,
        "upper_CI": upper_p * 100,
        "tau2": tau2,
        "I2": I2
    }


results = []

for group in ["Full RAMIE", "Hybrid RAMIE"]:

    res = subgroup_meta(
        robot_data[robot_data["robotic_extent"] == group]
    )

    res["group"] = group
    results.append(res)


subgroup_results = pd.DataFrame(results)

subgroup_results[
    ["group", "k", "pooled_AL",
     "lower_CI", "upper_CI",
     "I2", "tau2"]
]

In [ ]:
# Extract subgroup pooled estimates on logit scale
full = robot_data[robot_data["robotic_extent"] == "Full RAMIE"]
hybrid = robot_data[robot_data["robotic_extent"] == "Hybrid RAMIE"]


def get_reml_estimate(data):

    yi = data["yi"].values
    vi = data["vi"].values

    def reml_negloglik(tau2):
        wi = 1 / (vi + tau2)
        mu = np.sum(wi * yi) / np.sum(wi)

        return 0.5 * (
            np.sum(np.log(vi + tau2))
            + np.log(np.sum(wi))
            + np.sum(wi * (yi - mu)**2)
        )

    result = minimize_scalar(
        reml_negloglik,
        bounds=(0, 10),
        method="bounded"
    )

    tau2 = result.x
    wi = 1 / (vi + tau2)

    mu = np.sum(wi * yi) / np.sum(wi)
    se = np.sqrt(1 / np.sum(wi))

    return mu, se


mu_full, se_full = get_reml_estimate(full)
mu_hybrid, se_hybrid = get_reml_estimate(hybrid)

# Difference between subgroup pooled logits
difference = mu_full - mu_hybrid

se_difference = np.sqrt(
    se_full**2 + se_hybrid**2
)

z = difference / se_difference

from scipy.stats import norm

p_subgroup = 2 * (1 - norm.cdf(abs(z)))

print("Full RAMIE pooled logit =", round(mu_full, 4))
print("Hybrid RAMIE pooled logit =", round(mu_hybrid, 4))
print("Difference =", round(difference, 4))
print("Z =", round(z, 3))
print("P for subgroup difference =", round(p_subgroup, 4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import t

# Keep only Full and Hybrid RAMIE
meta_reg = al_data[
    al_data["study_id"].isin(
        df.loc[
            df["robotic_extent"].isin(["Full RAMIE", "Hybrid RAMIE"]),
            "study_id"
        ]
    )
].copy()

# Add robotic extent from original dataframe
meta_reg = meta_reg.merge(
    df[["study_id", "robotic_extent"]],
    on="study_id",
    how="left"
)

# Binary moderator:
# Hybrid RAMIE = 0 (reference)
# Full RAMIE   = 1
meta_reg["full_robotic"] = (
    meta_reg["robotic_extent"] == "Full RAMIE"
).astype(int)

# Effect size: logit proportion
meta_reg["yi"] = np.log(
    (meta_reg["al_event"] / meta_reg["number"]) /
    (1 - meta_reg["al_event"] / meta_reg["number"])
)

# Sampling variance
meta_reg["vi"] = (
    1 / meta_reg["al_event"] +
    1 / (meta_reg["number"] - meta_reg["al_event"])
)

# Design matrix: intercept + moderator
X = np.column_stack([
    np.ones(len(meta_reg)),
    meta_reg["full_robotic"].values
])

y = meta_reg["yi"].values
v = meta_reg["vi"].values

k = len(y)
p = X.shape[1]


# REML objective function
def reml_objective(tau2):

    W = np.diag(1 / (v + tau2))

    XtWX = X.T @ W @ X
    beta = np.linalg.solve(
        XtWX,
        X.T @ W @ y
    )

    residuals = y - X @ beta

    sign, logdet_X = np.linalg.slogdet(XtWX)

    return 0.5 * (
        np.sum(np.log(v + tau2))
        + logdet_X
        + residuals.T @ W @ residuals
    )


# Estimate tau² by REML
result = minimize_scalar(
    reml_objective,
    bounds=(0, 10),
    method="bounded"
)

tau2_meta = result.x

# Fit meta-regression
W = np.diag(1 / (v + tau2_meta))

XtWX_inv = np.linalg.inv(X.T @ W @ X)

beta = XtWX_inv @ X.T @ W @ y

residuals = y - X @ beta


# Knapp-Hartung adjustment
QE = residuals.T @ W @ residuals

df_resid = k - p

q = QE / df_resid

cov_beta_HK = q * XtWX_inv

se_HK = np.sqrt(np.diag(cov_beta_HK))


# Moderator coefficient = Full vs Hybrid
beta_mod = beta[1]
se_mod = se_HK[1]

t_value = beta_mod / se_mod

p_value = 2 * (
    1 - t.cdf(abs(t_value), df=df_resid)
)

tcrit = t.ppf(0.975, df=df_resid)

ci_lower = beta_mod - tcrit * se_mod
ci_upper = beta_mod + tcrit * se_mod


print("Number of cohorts =", k)
print("Residual df =", df_resid)
print("REML tau² =", round(tau2_meta, 4))
print()
print("Full vs Hybrid coefficient =", round(beta_mod, 4))
print("HK SE =", round(se_mod, 4))
print("t =", round(t_value, 3))
print("95% CI =", round(ci_lower, 4), "to", round(ci_upper, 4))
print("P for moderator =", round(p_value, 4))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit

# =========================================================
# DATA
# =========================================================

plot_data = robot_data.copy()

plot_data["proportion"] = (
    plot_data["al_event"] / plot_data["number"]
)

# Individual-study CI on logit scale
plot_data["ci_low_logit"] = (
    plot_data["yi"] - 1.96 * np.sqrt(plot_data["vi"])
)

plot_data["ci_high_logit"] = (
    plot_data["yi"] + 1.96 * np.sqrt(plot_data["vi"])
)

plot_data["ci_lower"] = expit(plot_data["ci_low_logit"])
plot_data["ci_upper"] = expit(plot_data["ci_high_logit"])


summary = {
    "Full RAMIE": {
        "pooled": 0.12957295,
        "lower": 0.07342632,
        "upper": 0.21852737,
        "I2": 79.121957
    },
    "Hybrid RAMIE": {
        "pooled": 0.07385222,
        "lower": 0.05846646,
        "upper": 0.09288738,
        "I2": 0.0
    }
}

groups = ["Full RAMIE", "Hybrid RAMIE"]


# =========================================================
# CREATE ROW STRUCTURE
# =========================================================

rows = []

for group in groups:

    rows.append({
        "type": "group",
        "group": group
    })

    temp = plot_data[
        plot_data["robotic_extent"] == group
    ]

    for _, r in temp.iterrows():

        rows.append({
            "type": "study",
            "group": group,
            "study": r["study_id"],
            "events": int(r["al_event"]),
            "total": int(r["number"]),
            "prop": r["proportion"],
            "low": r["ci_lower"],
            "high": r["ci_upper"]
        })

    rows.append({
        "type": "pooled",
        "group": group
    })

    rows.append({
        "type": "space"
    })


# =========================================================
# FIGURE — FOUR SEPARATE PANELS
# =========================================================

fig = plt.figure(figsize=(14, 11))

gs = fig.add_gridspec(
    1, 4,
    width_ratios=[3.2, 4.4, 1.5, 3.0],
    wspace=0.03
)

ax_study = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1])
ax_events = fig.add_subplot(gs[0, 2])
ax_effect = fig.add_subplot(gs[0, 3])

axes = [ax_study, ax_forest, ax_events, ax_effect]

N = len(rows)

for ax in axes:
    ax.set_ylim(-3, N + 2)
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)


# =========================================================
# HEADERS
# =========================================================

header_y = N + 1

ax_study.text(
    0, header_y,
    "Study",
    fontsize=11,
    fontweight="bold",
    va="center"
)

ax_events.text(
    0.5, header_y,
    "Events/Total",
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center"
)

ax_effect.text(
    0.5, header_y,
    "AL rate [95% CI]",
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center"
)

# horizontal header lines
for ax in axes:
    ax.axhline(
        N + 0.45,
        linewidth=0.7
    )


# =========================================================
# FOREST AXIS
# =========================================================

ax_forest.set_xlim(0, 0.65)

ticks = [0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

ax_forest.set_xticks(ticks)

ax_forest.set_xticklabels(
    ["0", "10", "20", "30", "40", "50", "60"],
    fontsize=9
)

ax_forest.set_xlabel(
    "Anastomotic leak rate (%)",
    fontsize=10
)

# Overall pooled estimate reference line
ax_forest.axvline(
    0.1002,
    linestyle="--",
    linewidth=1,
    alpha=0.7
)


# Text axes
ax_study.set_xlim(0, 1)
ax_events.set_xlim(0, 1)
ax_effect.set_xlim(0, 1)

ax_study.set_xticks([])
ax_events.set_xticks([])
ax_effect.set_xticks([])


# =========================================================
# DRAW ROWS
# =========================================================

y = N

for row in rows:

    if row["type"] == "group":

        ax_study.text(
            0, y,
            row["group"],
            fontsize=11,
            fontweight="bold",
            va="center"
        )

    elif row["type"] == "study":

        # Study name
        ax_study.text(
            0.03, y,
            row["study"],
            fontsize=9.5,
            va="center"
        )

        # CI line
        ax_forest.plot(
            [row["low"], row["high"]],
            [y, y],
            linewidth=1.3
        )

        # Square
        ax_forest.scatter(
            row["prop"],
            y,
            marker="s",
            s=42,
            zorder=3
        )

        # Events/Total
        ax_events.text(
            0.5, y,
            f'{row["events"]}/{row["total"]}',
            fontsize=9.5,
            ha="center",
            va="center"
        )

        # Effect estimate
        ax_effect.text(
            0.5, y,
            f'{row["prop"]*100:.1f}% '
            f'[{row["low"]*100:.1f}–{row["high"]*100:.1f}]',
            fontsize=9.5,
            ha="center",
            va="center"
        )

    elif row["type"] == "pooled":

        group = row["group"]

        pooled = summary[group]["pooled"]
        low = summary[group]["lower"]
        high = summary[group]["upper"]
        I2 = summary[group]["I2"]

        # Diamond
        diamond_x = [
            low,
            pooled,
            high,
            pooled,
            low
        ]

        diamond_y = [
            y,
            y + 0.28,
            y,
            y - 0.28,
            y
        ]

        ax_forest.fill(
            diamond_x,
            diamond_y,
            alpha=0.8
        )

        ax_study.text(
            0, y,
            f"{group} pooled",
            fontsize=10,
            fontweight="bold",
            va="center"
        )

        ax_effect.text(
            0.5, y,
            f'{pooled*100:.2f}% '
            f'[{low*100:.2f}–{high*100:.2f}]',
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center"
        )

        # I² below pooled estimate
        ax_events.text(
            0.5, y,
            f'I² = {I2:.1f}%',
            fontsize=9.5,
            fontweight="bold",
            ha="center",
            va="center"
        )

    y -= 1


# =========================================================
# SUBGROUP TEST
# =========================================================

ax_study.text(
    0,
    -0.7,
    "Test for subgroup difference: P = 0.069",
    fontsize=10,
    fontweight="bold"
)

ax_study.text(
    0,
    -1.7,
    "Random-effects model (REML); Knapp–Hartung adjustment",
    fontsize=9
)


# =========================================================
# FINAL FORMATTING
# =========================================================

plt.subplots_adjust(
    left=0.05,
    right=0.98,
    top=0.96,
    bottom=0.10
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# ============================================================
# FINAL PUBLICATION-READY SUBGROUP FOREST PLOT
# Full RAMIE vs Hybrid RAMIE
# ============================================================

plot_df = al_data[
    al_data["robotic_extent"].isin(["Full RAMIE", "Hybrid RAMIE"])
].copy()

# ------------------------------------------------------------
# 1. Exact 95% CI for individual study proportions
#    Clopper-Pearson
# ------------------------------------------------------------
alpha = 0.05

def exact_ci(x, n):
    if x == 0:
        lower = 0
    else:
        lower = beta.ppf(alpha / 2, x, n - x + 1)

    if x == n:
        upper = 1
    else:
        upper = beta.ppf(1 - alpha / 2, x + 1, n - x)

    return lower, upper


cis = plot_df.apply(
    lambda row: exact_ci(int(row["al_event"]), int(row["number"])),
    axis=1
)

plot_df["ci_lower"] = [x[0] for x in cis]
plot_df["ci_upper"] = [x[1] for x in cis]

# ------------------------------------------------------------
# 2. Random-effects weights using REML tau²
# ------------------------------------------------------------
tau2_plot = 0.3165

plot_df["re_weight"] = 1 / (plot_df["variance"] + tau2_plot)
plot_df["weight_percent"] = (
    plot_df["re_weight"] / plot_df["re_weight"].sum() * 100
)

# ------------------------------------------------------------
# 3. Subgroup summary estimates
#    Results obtained previously
# ------------------------------------------------------------
subgroup_results = {
    "Full RAMIE": {
        "pooled": 0.12957295,
        "lower": 0.07342632,
        "upper": 0.21852737,
        "I2": 79.121957
    },

    "Hybrid RAMIE": {
        "pooled": 0.07385222,
        "lower": 0.05846646,
        "upper": 0.09288738,
        "I2": 0.0
    }
}

# Overall result
overall_pooled = 0.1002
overall_lower = 0.0734
overall_upper = 0.1352

# Prediction interval
prediction_lower = 0.0310
prediction_upper = 0.2792

# Moderator test
p_subgroup = 0.0687


# ------------------------------------------------------------
# 4. Build row positions
# ------------------------------------------------------------
groups = ["Full RAMIE", "Hybrid RAMIE"]

rows = []
y = 0

for group in groups:

    rows.append(("header", group, None, y))
    y += 1

    temp = plot_df[plot_df["robotic_extent"] == group]

    for idx, row in temp.iterrows():
        rows.append(("study", group, idx, y))
        y += 1

    rows.append(("pooled", group, None, y))
    y += 1.2

    # space between groups
    y += 0.5


# Overall pooled row
overall_y = y
y += 1.2

prediction_y = y
y += 1.2

heterogeneity_y = y
y += 0.9

subgroup_test_y = y

total_height = y + 1


# ------------------------------------------------------------
# 5. Create figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 12))

ax.set_xlim(-0.095, 0.42)
ax.set_ylim(total_height, -1)

# Remove axes/spines
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(
    left=False,
    bottom=False,
    labelleft=False,
    labelbottom=False
)


# ------------------------------------------------------------
# 6. Column positions
# ------------------------------------------------------------
x_study = -0.092
x_events = 0.225
x_effect = 0.305
x_weight = 0.395


# ------------------------------------------------------------
# 7. Column headings
# ------------------------------------------------------------
header_y = -0.55

ax.text(
    x_study, header_y,
    "Study",
    fontsize=12,
    fontweight="bold",
    ha="left"
)

ax.text(
    x_events, header_y,
    "Events/Total",
    fontsize=12,
    fontweight="bold",
    ha="center"
)

ax.text(
    x_effect, header_y,
    "AL rate [95% CI]",
    fontsize=12,
    fontweight="bold",
    ha="center"
)

ax.text(
    x_weight, header_y,
    "Weight",
    fontsize=12,
    fontweight="bold",
    ha="center"
)

# line below header
ax.hlines(
    -0.25,
    xmin=-0.095,
    xmax=0.415,
    linewidth=0.8
)


# ------------------------------------------------------------
# 8. Reference line = overall pooled estimate
# ------------------------------------------------------------
ax.axvline(
    overall_pooled,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)


# ------------------------------------------------------------
# 9. Plot study rows and subgroup pooled diamonds
# ------------------------------------------------------------
for row_type, group, idx, ypos in rows:

    # ---- subgroup heading ----
    if row_type == "header":

        ax.text(
            x_study,
            ypos,
            group,
            fontsize=12,
            fontweight="bold",
            va="center"
        )

    # ---- individual study ----
    elif row_type == "study":

        r = plot_df.loc[idx]

        p = r["proportion"]
        lo = r["ci_lower"]
        hi = r["ci_upper"]

        weight = r["weight_percent"]

        # CI line
        ax.hlines(
            ypos,
            lo,
            hi,
            linewidth=1.3
        )

        # square proportional to weight
        marker_size = 4 + weight * 1.25

        ax.plot(
            p,
            ypos,
            marker="s",
            markersize=marker_size
        )

        # study label
        ax.text(
            x_study,
            ypos,
            str(r["study_id"]),
            fontsize=10,
            va="center"
        )

        # Events / total
        ax.text(
            x_events,
            ypos,
            f'{int(r["al_event"])}/{int(r["number"])}',
            fontsize=10,
            ha="center",
            va="center"
        )

        # Effect + CI
        ax.text(
            x_effect,
            ypos,
            f'{p*100:.1f}% [{lo*100:.1f}–{hi*100:.1f}]',
            fontsize=10,
            ha="center",
            va="center"
        )

        # Weight
        ax.text(
            x_weight,
            ypos,
            f'{weight:.1f}%',
            fontsize=10,
            ha="center",
            va="center"
        )


    # ---- subgroup pooled estimate ----
    elif row_type == "pooled":

        res = subgroup_results[group]

        pooled = res["pooled"]
        lo = res["lower"]
        hi = res["upper"]
        I2 = res["I2"]

        diamond_height = 0.25

        diamond_x = [
            lo,
            pooled,
            hi,
            pooled
        ]

        diamond_y = [
            ypos,
            ypos - diamond_height,
            ypos,
            ypos + diamond_height
        ]

        ax.fill(
            diamond_x,
            diamond_y,
            alpha=0.8
        )

        ax.text(
            x_study,
            ypos,
            f"{group} pooled",
            fontsize=11,
            fontweight="bold",
            va="center"
        )

        ax.text(
            x_events,
            ypos,
            f"I² = {I2:.1f}%",
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center"
        )

        ax.text(
            x_effect,
            ypos,
            f'{pooled*100:.2f}% [{lo*100:.2f}–{hi*100:.2f}]',
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center"
        )


# ------------------------------------------------------------
# 10. Overall pooled estimate
# ------------------------------------------------------------
diamond_height = 0.28

diamond_x = [
    overall_lower,
    overall_pooled,
    overall_upper,
    overall_pooled
]

diamond_y = [
    overall_y,
    overall_y - diamond_height,
    overall_y,
    overall_y + diamond_height
]

ax.fill(
    diamond_x,
    diamond_y,
    alpha=0.85
)

ax.text(
    x_study,
    overall_y,
    "Overall random-effects model",
    fontsize=11,
    fontweight="bold",
    va="center"
)

ax.text(
    x_effect,
    overall_y,
    f"{overall_pooled*100:.2f}% "
    f"[{overall_lower*100:.2f}–{overall_upper*100:.2f}]",
    fontsize=10,
    fontweight="bold",
    ha="center",
    va="center"
)


# ------------------------------------------------------------
# 11. Prediction interval
# ------------------------------------------------------------
ax.hlines(
    prediction_y,
    prediction_lower,
    prediction_upper,
    linewidth=2
)

ax.plot(
    [prediction_lower, prediction_upper],
    [prediction_y, prediction_y],
    "|",
    markersize=10
)

ax.plot(
    overall_pooled,
    prediction_y,
    marker="D",
    markersize=5
)

ax.text(
    x_study,
    prediction_y,
    "Prediction interval",
    fontsize=10,
    va="center"
)

ax.text(
    x_effect,
    prediction_y,
    f"{prediction_lower*100:.2f}%–{prediction_upper*100:.2f}%",
    fontsize=10,
    ha="center",
    va="center"
)


# ------------------------------------------------------------
# 12. Overall heterogeneity
# ------------------------------------------------------------
ax.text(
    x_study,
    heterogeneity_y,
    "Overall heterogeneity: "
    "I² = 62.0%; τ² = 0.3165",
    fontsize=10,
    va="center"
)


# ------------------------------------------------------------
# 13. Test for subgroup difference
# ------------------------------------------------------------
ax.text(
    x_study,
    subgroup_test_y,
    f"Test for subgroup difference: "
    f"P = {p_subgroup:.4f}",
    fontsize=10,
    va="center"
)


# ------------------------------------------------------------
# 14. X-axis labels
# ------------------------------------------------------------
ticks = [0, 0.05, 0.10, 0.20, 0.30, 0.40]

for tick in ticks:
    ax.text(
        tick,
        total_height - 0.15,
        f"{tick*100:.0f}%",
        ha="center",
        fontsize=9
    )

ax.text(
    0.20,
    total_height + 0.35,
    "Anastomotic leak rate",
    ha="center",
    fontsize=10
)


# ------------------------------------------------------------
# 15. Final layout
# ------------------------------------------------------------
plt.tight_layout()

# High-resolution PNG
plt.savefig(
    "RAMIE_subgroup_forest_final.png",
    dpi=600,
    bbox_inches="tight"
)

# Vector PDF
plt.savefig(
    "RAMIE_subgroup_forest_final.pdf",
    bbox_inches="tight"
)

# TIFF for journal submission
plt.savefig(
    "RAMIE_subgroup_forest_final.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
df = pd.read_excel("ICG(1).xlsx", header=1)
df.head()

In [ ]:
# Add robotic_extent from the original dataframe to al_data
al_data["robotic_extent"] = df.loc[al_data.index, "robotic_extent"]

# Check
al_data[["study_id", "robotic_extent", "number", "al_event"]]

In [ ]:
print(al_data["robotic_extent"].value_counts(dropna=False))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# ============================================================
# 1. ADD ROBOTIC EXTENT TO al_data
# ============================================================

# robotic_extent exists in the original dataframe (df)
al_data["robotic_extent"] = df.loc[al_data.index, "robotic_extent"].values

print("Robotic extent:")
print(al_data["robotic_extent"].value_counts(dropna=False))


# ============================================================
# 2. SELECT FULL AND HYBRID RAMIE
# ============================================================

plot_df = al_data[
    al_data["robotic_extent"].isin(["Full RAMIE", "Hybrid RAMIE"])
].copy()

print("\nNumber included in subgroup forest plot:", len(plot_df))


# ============================================================
# 3. EXACT 95% CI FOR EACH STUDY
# Clopper-Pearson confidence intervals
# ============================================================

alpha = 0.05

def exact_ci(x, n):
    if x == 0:
        lower = 0
    else:
        lower = beta.ppf(alpha / 2, x, n - x + 1)

    if x == n:
        upper = 1
    else:
        upper = beta.ppf(1 - alpha / 2, x + 1, n - x)

    return lower, upper


cis = plot_df.apply(
    lambda row: exact_ci(
        int(row["al_event"]),
        int(row["number"])
    ),
    axis=1
)

plot_df["ci_lower"] = [x[0] for x in cis]
plot_df["ci_upper"] = [x[1] for x in cis]


# ============================================================
# 4. RANDOM-EFFECTS WEIGHTS
# Overall REML tau² obtained previously
# ============================================================

tau2_plot = 0.3165

plot_df["re_weight"] = 1 / (
    plot_df["variance"] + tau2_plot
)

plot_df["weight_percent"] = (
    plot_df["re_weight"]
    / plot_df["re_weight"].sum()
    * 100
)


# ============================================================
# 5. SUBGROUP RESULTS OBTAINED PREVIOUSLY
# ============================================================

subgroup_results = {

    "Full RAMIE": {
        "pooled": 0.12957295,
        "lower": 0.07342632,
        "upper": 0.21852737,
        "I2": 79.121957
    },

    "Hybrid RAMIE": {
        "pooled": 0.07385222,
        "lower": 0.05846646,
        "upper": 0.09288738,
        "I2": 0.0
    }
}


# Overall REML + Hartung-Knapp results
overall_pooled = 0.1002
overall_lower = 0.0734
overall_upper = 0.1352

# Prediction interval
prediction_lower = 0.0310
prediction_upper = 0.2792

# Meta-regression / moderator test
p_subgroup = 0.0687


# ============================================================
# 6. CREATE ROW POSITIONS
# ============================================================

groups = ["Full RAMIE", "Hybrid RAMIE"]

rows = []

y = 0

for group in groups:

    # subgroup heading
    rows.append(("header", group, None, y))
    y += 1

    temp = plot_df[
        plot_df["robotic_extent"] == group
    ]

    # individual studies
    for idx, row in temp.iterrows():
        rows.append(("study", group, idx, y))
        y += 1

    # subgroup pooled result
    rows.append(("pooled", group, None, y))
    y += 1

    # small gap between subgroups
    y += 0.7


# Overall rows
overall_y = y
y += 1.2

prediction_y = y
y += 1.2

heterogeneity_y = y
y += 0.9

subgroup_test_y = y
y += 1.0

total_height = y


# ============================================================
# 7. CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(16, 12))

# Enough room for study names and result columns
ax.set_xlim(-0.10, 0.43)

ax.set_ylim(
    total_height + 0.7,
    -1
)


# Remove normal plot frame
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(
    left=False,
    bottom=False,
    labelleft=False,
    labelbottom=False
)


# ============================================================
# 8. COLUMN POSITIONS
# ============================================================

x_study = -0.098

x_events = 0.235

x_effect = 0.325

x_weight = 0.405


# ============================================================
# 9. HEADERS
# ============================================================

header_y = -0.55

ax.text(
    x_study,
    header_y,
    "Study",
    fontsize=12,
    fontweight="bold",
    ha="left"
)

ax.text(
    x_events,
    header_y,
    "Events/Total",
    fontsize=12,
    fontweight="bold",
    ha="center"
)

ax.text(
    x_effect,
    header_y,
    "AL rate [95% CI]",
    fontsize=12,
    fontweight="bold",
    ha="center"
)

ax.text(
    x_weight,
    header_y,
    "Weight",
    fontsize=12,
    fontweight="bold",
    ha="center"
)


# horizontal line under headings
ax.hlines(
    -0.25,
    xmin=-0.10,
    xmax=0.425,
    linewidth=0.8
)


# ============================================================
# 10. OVERALL POOLED REFERENCE LINE
# ============================================================

ax.axvline(
    overall_pooled,
    linestyle="--",
    linewidth=1,
    alpha=0.5
)


# ============================================================
# 11. INDIVIDUAL STUDIES + SUBGROUP RESULTS
# ============================================================

for row_type, group, idx, ypos in rows:

    # --------------------------------------------------------
    # SUBGROUP HEADING
    # --------------------------------------------------------

    if row_type == "header":

        ax.text(
            x_study,
            ypos,
            group,
            fontsize=12,
            fontweight="bold",
            va="center"
        )


    # --------------------------------------------------------
    # INDIVIDUAL STUDY
    # --------------------------------------------------------

    elif row_type == "study":

        r = plot_df.loc[idx]

        p = r["proportion"]

        lo = r["ci_lower"]

        hi = r["ci_upper"]

        weight = r["weight_percent"]


        # Confidence interval
        ax.hlines(
            ypos,
            lo,
            hi,
            linewidth=1.3
        )


        # Square size according to random-effects weight
        marker_size = 4 + weight * 1.1

        ax.plot(
            p,
            ypos,
            marker="s",
            markersize=marker_size
        )


        # Study name
        ax.text(
            x_study,
            ypos,
            str(r["study_id"]),
            fontsize=10,
            va="center"
        )


        # Events / Total
        ax.text(
            x_events,
            ypos,
            f'{int(r["al_event"])}/{int(r["number"])}',
            fontsize=10,
            ha="center",
            va="center"
        )


        # AL rate + CI
        ax.text(
            x_effect,
            ypos,
            f'{p*100:.1f}% '
            f'[{lo*100:.1f}–{hi*100:.1f}]',
            fontsize=10,
            ha="center",
            va="center"
        )


        # Weight
        ax.text(
            x_weight,
            ypos,
            f'{weight:.1f}%',
            fontsize=10,
            ha="center",
            va="center"
        )


    # --------------------------------------------------------
    # SUBGROUP POOLED RESULT
    # --------------------------------------------------------

    elif row_type == "pooled":

        res = subgroup_results[group]

        pooled = res["pooled"]

        lo = res["lower"]

        hi = res["upper"]

        I2 = res["I2"]


        diamond_height = 0.25


        diamond_x = [
            lo,
            pooled,
            hi,
            pooled
        ]


        diamond_y = [
            ypos,
            ypos - diamond_height,
            ypos,
            ypos + diamond_height
        ]


        ax.fill(
            diamond_x,
            diamond_y,
            alpha=0.8
        )


        ax.text(
            x_study,
            ypos,
            f"{group} pooled",
            fontsize=11,
            fontweight="bold",
            va="center"
        )


        ax.text(
            x_events,
            ypos,
            f"I² = {I2:.1f}%",
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center"
        )


        ax.text(
            x_effect,
            ypos,
            f'{pooled*100:.2f}% '
            f'[{lo*100:.2f}–{hi*100:.2f}]',
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center"
        )


# ============================================================
# 12. OVERALL RANDOM-EFFECTS MODEL
# ============================================================

diamond_height = 0.28


diamond_x = [
    overall_lower,
    overall_pooled,
    overall_upper,
    overall_pooled
]


diamond_y = [
    overall_y,
    overall_y - diamond_height,
    overall_y,
    overall_y + diamond_height
]


ax.fill(
    diamond_x,
    diamond_y,
    alpha=0.85
)


ax.text(
    x_study,
    overall_y,
    "Overall random-effects model",
    fontsize=11,
    fontweight="bold",
    va="center"
)


ax.text(
    x_effect,
    overall_y,
    f"{overall_pooled*100:.2f}% "
    f"[{overall_lower*100:.2f}–{overall_upper*100:.2f}]",
    fontsize=10,
    fontweight="bold",
    ha="center",
    va="center"
)


ax.text(
    x_weight,
    overall_y,
    "100%",
    fontsize=10,
    fontweight="bold",
    ha="center",
    va="center"
)


# ============================================================
# 13. PREDICTION INTERVAL
# ============================================================

ax.hlines(
    prediction_y,
    prediction_lower,
    prediction_upper,
    linewidth=2
)


ax.plot(
    [prediction_lower, prediction_upper],
    [prediction_y, prediction_y],
    "|",
    markersize=10
)


ax.plot(
    overall_pooled,
    prediction_y,
    marker="D",
    markersize=5
)


ax.text(
    x_study,
    prediction_y,
    "Prediction interval",
    fontsize=10,
    va="center"
)


ax.text(
    x_effect,
    prediction_y,
    f"{prediction_lower*100:.2f}%–"
    f"{prediction_upper*100:.2f}%",
    fontsize=10,
    ha="center",
    va="center"
)


# ============================================================
# 14. HETEROGENEITY
# ============================================================

ax.text(
    x_study,
    heterogeneity_y,
    "Overall heterogeneity: "
    "I² = 62.0%; τ² = 0.3165",
    fontsize=10,
    va="center"
)


# ============================================================
# 15. TEST FOR SUBGROUP DIFFERENCE
# ============================================================

ax.text(
    x_study,
    subgroup_test_y,
    f"Test for subgroup difference: P = {p_subgroup:.4f}",
    fontsize=10,
    va="center"
)


# ============================================================
# 16. X-AXIS
# ============================================================

ticks = [
    0,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40
]


for tick in ticks:

    ax.text(
        tick,
        total_height + 0.15,
        f"{tick*100:.0f}%",
        ha="center",
        fontsize=9
    )


ax.text(
    0.20,
    total_height + 0.65,
    "Anastomotic leak rate",
    ha="center",
    fontsize=10
)


# ============================================================
# 17. FINAL LAYOUT
# ============================================================

plt.tight_layout()


# ============================================================
# 18. SAVE FIGURES
# ============================================================

plt.savefig(
    "RAMIE_subgroup_forest_final.png",
    dpi=600,
    bbox_inches="tight"
)


plt.savefig(
    "RAMIE_subgroup_forest_final.pdf",
    bbox_inches="tight"
)


plt.savefig(
    "RAMIE_subgroup_forest_final.tiff",
    dpi=600,
    bbox_inches="tight"
)


plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# PUBLICATION-READY SUBGROUP FOREST PLOT
# Full RAMIE vs Hybrid RAMIE
# ============================================================

groups = ["Full RAMIE", "Hybrid RAMIE"]

subgroup_results = {
    "Full RAMIE": {
        "pooled": 0.12957295,
        "lower": 0.07342632,
        "upper": 0.21852737,
        "I2": 79.121957
    },
    "Hybrid RAMIE": {
        "pooled": 0.07385222,
        "lower": 0.05846646,
        "upper": 0.09288738,
        "I2": 0.0
    }
}

# Overall results from previous analysis
overall_pooled = 0.1002
overall_lower = 0.0734
overall_upper = 0.1352

prediction_lower = 0.0310
prediction_upper = 0.2792

overall_I2 = 62.0
overall_tau2 = 0.3165

p_subgroup = 0.0687


# ============================================================
# CREATE ROW STRUCTURE
# ============================================================

rows = []
y = 0

for group in groups:

    rows.append(("header", group, None, y))
    y += 1

    temp = plot_df[
        plot_df["robotic_extent"] == group
    ]

    for idx in temp.index:
        rows.append(("study", group, idx, y))
        y += 1

    rows.append(("pooled", group, None, y))
    y += 1.5


overall_y = y
y += 1.3

prediction_y = y
y += 1.2

heterogeneity_y = y
y += 0.9

subgroup_test_y = y
y += 1

total_height = y


# ============================================================
# CREATE SEPARATE PANELS
# ============================================================

fig = plt.figure(figsize=(16, 12))

gs = GridSpec(
    1, 5,
    width_ratios=[2.8, 4.0, 1.25, 2.35, 1.0],
    wspace=0.03
)

ax_study = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1])
ax_events = fig.add_subplot(gs[0, 2])
ax_effect = fig.add_subplot(gs[0, 3])
ax_weight = fig.add_subplot(gs[0, 4])


axes = [
    ax_study,
    ax_forest,
    ax_events,
    ax_effect,
    ax_weight
]


# ============================================================
# COMMON AXIS SETTINGS
# ============================================================

for ax in axes:

    ax.set_ylim(total_height, -1)

    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)


# Remove x axes from text panels
for ax in [
    ax_study,
    ax_events,
    ax_effect,
    ax_weight
]:
    ax.set_xticks([])


# ============================================================
# HEADERS
# ============================================================

header_y = -0.55

ax_study.text(
    0,
    header_y,
    "Study",
    fontsize=12,
    fontweight="bold",
    va="center"
)

ax_events.text(
    0.5,
    header_y,
    "Events/Total",
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="center"
)

ax_effect.text(
    0.5,
    header_y,
    "AL rate [95% CI]",
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="center"
)

ax_weight.text(
    0.5,
    header_y,
    "Weight",
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="center"
)


# Header lines
for ax in axes:
    ax.axhline(
        -0.15,
        linewidth=0.8,
        alpha=0.7
    )


# ============================================================
# FOREST AXIS
# ============================================================

ax_forest.set_xlim(0, 0.50)

ax_forest.set_xticks(
    [0, 0.10, 0.20, 0.30, 0.40, 0.50]
)

ax_forest.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%", "50%"],
    fontsize=9
)

ax_forest.set_xlabel(
    "Anastomotic leak rate",
    fontsize=10
)

# Overall pooled estimate reference line
ax_forest.axvline(
    overall_pooled,
    linestyle="--",
    linewidth=1,
    alpha=0.5
)


# ============================================================
# PLOT ROWS
# ============================================================

for row_type, group, idx, ypos in rows:

    # --------------------------------------------------------
    # SUBGROUP HEADER
    # --------------------------------------------------------

    if row_type == "header":

        ax_study.text(
            0,
            ypos,
            group,
            fontsize=11,
            fontweight="bold",
            va="center"
        )


    # --------------------------------------------------------
    # INDIVIDUAL STUDY
    # --------------------------------------------------------

    elif row_type == "study":

        r = plot_df.loc[idx]

        p = float(r["proportion"])
        lo = float(r["ci_lower"])
        hi = float(r["ci_upper"])
        weight = float(r["weight_percent"])


        # Study name
        ax_study.text(
            0.02,
            ypos,
            str(r["study_id"]),
            fontsize=9.5,
            va="center"
        )


        # Confidence interval
        ax_forest.hlines(
            ypos,
            lo,
            hi,
            linewidth=1.3
        )


        # Weight-proportional square
        marker_size = 4 + weight * 1.15

        ax_forest.plot(
            p,
            ypos,
            marker="s",
            markersize=marker_size
        )


        # Events / Total
        ax_events.text(
            0.5,
            ypos,
            f'{int(r["al_event"])}/{int(r["number"])}',
            fontsize=9.5,
            ha="center",
            va="center"
        )


        # Effect estimate
        ax_effect.text(
            0.5,
            ypos,
            f'{p*100:.1f}% '
            f'[{lo*100:.1f}–{hi*100:.1f}]',
            fontsize=9.5,
            ha="center",
            va="center"
        )


        # Weight
        ax_weight.text(
            0.5,
            ypos,
            f'{weight:.1f}%',
            fontsize=9.5,
            ha="center",
            va="center"
        )


    # --------------------------------------------------------
    # SUBGROUP POOLED RESULT
    # --------------------------------------------------------

    elif row_type == "pooled":

        res = subgroup_results[group]

        pooled = res["pooled"]
        lo = res["lower"]
        hi = res["upper"]
        I2 = res["I2"]


        # Diamond
        h = 0.25

        ax_forest.fill(
            [lo, pooled, hi, pooled],
            [ypos, ypos - h, ypos, ypos + h],
            alpha=0.8
        )


        # Label
        ax_study.text(
            0,
            ypos,
            f"{group} pooled",
            fontsize=10.5,
            fontweight="bold",
            va="center"
        )


        # I2
        ax_events.text(
            0.5,
            ypos,
            f"I² = {I2:.1f}%",
            fontsize=9.5,
            fontweight="bold",
            ha="center",
            va="center"
        )


        # Pooled estimate
        ax_effect.text(
            0.5,
            ypos,
            f'{pooled*100:.2f}% '
            f'[{lo*100:.2f}–{hi*100:.2f}]',
            fontsize=9.5,
            fontweight="bold",
            ha="center",
            va="center"
        )


# ============================================================
# OVERALL RANDOM-EFFECTS MODEL
# ============================================================

h = 0.27

ax_forest.fill(
    [
        overall_lower,
        overall_pooled,
        overall_upper,
        overall_pooled
    ],
    [
        overall_y,
        overall_y - h,
        overall_y,
        overall_y + h
    ],
    alpha=0.85
)


ax_study.text(
    0,
    overall_y,
    "Overall random-effects model",
    fontsize=10.5,
    fontweight="bold",
    va="center"
)


ax_effect.text(
    0.5,
    overall_y,
    f"{overall_pooled*100:.2f}% "
    f"[{overall_lower*100:.2f}–{overall_upper*100:.2f}]",
    fontsize=9.5,
    fontweight="bold",
    ha="center",
    va="center"
)


ax_weight.text(
    0.5,
    overall_y,
    "100%",
    fontsize=9.5,
    fontweight="bold",
    ha="center",
    va="center"
)


# ============================================================
# PREDICTION INTERVAL
# ============================================================

ax_study.text(
    0,
    prediction_y,
    "Prediction interval",
    fontsize=9.5,
    va="center"
)


ax_forest.hlines(
    prediction_y,
    prediction_lower,
    prediction_upper,
    linewidth=2
)

ax_forest.plot(
    [prediction_lower, prediction_upper],
    [prediction_y, prediction_y],
    "|",
    markersize=9
)

ax_forest.plot(
    overall_pooled,
    prediction_y,
    marker="D",
    markersize=5
)


ax_effect.text(
    0.5,
    prediction_y,
    f"{prediction_lower*100:.2f}%–"
    f"{prediction_upper*100:.2f}%",
    fontsize=9.5,
    ha="center",
    va="center"
)


# ============================================================
# OVERALL HETEROGENEITY
# ============================================================

ax_study.text(
    0,
    heterogeneity_y,
    f"Overall heterogeneity: "
    f"I² = {overall_I2:.1f}%; "
    f"τ² = {overall_tau2:.4f}",
    fontsize=9.5,
    va="center"
)


# ============================================================
# TEST FOR SUBGROUP DIFFERENCE
# ============================================================

ax_study.text(
    0,
    subgroup_test_y,
    f"Test for subgroup difference: P = {p_subgroup:.4f}",
    fontsize=9.5,
    va="center"
)


# ============================================================
# PANEL LIMITS
# ============================================================

ax_study.set_xlim(0, 1)

ax_events.set_xlim(0, 1)

ax_effect.set_xlim(0, 1)

ax_weight.set_xlim(0, 1)


# ============================================================
# FINAL LAYOUT
# ============================================================

plt.subplots_adjust(
    left=0.04,
    right=0.98,
    top=0.98,
    bottom=0.08,
    wspace=0.03
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "RAMIE_subgroup_forest_publication.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "RAMIE_subgroup_forest_publication.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "RAMIE_subgroup_forest_publication.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
df["anastomotic_technique"].value_counts(dropna=False)

In [ ]:
# Check cleaned anastomotic technique categories

print(df["anastomotic_technique"].value_counts(dropna=False))

print("\nNumber of unique categories:")
print(df["anastomotic_technique"].nunique(dropna=False))

In [ ]:
print(df["anastomotic_technique"].value_counts(dropna=False))

print("\nNumber of unique categories:")
print(df["anastomotic_technique"].nunique(dropna=False))

In [ ]:
df = pd.read_excel("ICG2.xlsx", header=1)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df["anastomotic_technique"].value_counts(dropna=False))

In [ ]:
df = pd.read_excel("ICG2.xlsx", header=1)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(df["anastomotic_technique"].value_counts(dropna=False))

In [ ]:
# Rebuild analysis dataset from the updated Excel file

al_data = df[
    [
        "study_id",
        "number",
        "al_event",
        "al_rate",
        "robotic_extent",
        "procedure",
        "anastomotic_technique"
    ]
].copy()

# Check the technique groups and AL data
print(
    al_data[
        ["study_id", "anastomotic_technique",
         "number", "al_event", "al_rate"]
    ].to_string(index=False)
)

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import t

# ============================================================
# Anastomotic technique subgroup analysis
# ============================================================

tech_data = al_data[
    al_data["anastomotic_technique"].isin([
        "circular-stapled",
        "linear-stapled",
        "hand-sewn-stapled"
    ])
].copy()

# Calculate proportion and logit effect size
tech_data["proportion"] = (
    tech_data["al_event"] / tech_data["number"]
)

tech_data["logit_p"] = np.log(
    tech_data["proportion"] /
    (1 - tech_data["proportion"])
)

# Within-study variance of logit proportion
tech_data["variance"] = (
    1 / tech_data["al_event"] +
    1 / (tech_data["number"] - tech_data["al_event"])
)


# ------------------------------------------------------------
# REML function
# ------------------------------------------------------------

def reml_meta(data):

    y = data["logit_p"].values
    v = data["variance"].values
    k = len(data)

    def neg_reml(tau2):
        w = 1 / (v + tau2)
        mu = np.sum(w * y) / np.sum(w)

        return 0.5 * (
            np.sum(np.log(v + tau2))
            + np.log(np.sum(w))
            + np.sum(w * (y - mu)**2)
        )

    result = minimize_scalar(
        neg_reml,
        bounds=(0, 10),
        method="bounded"
    )

    tau2 = result.x

    w = 1 / (v + tau2)

    pooled_logit = np.sum(w * y) / np.sum(w)

    # Standard error
    se = np.sqrt(1 / np.sum(w))

    # Hartung-Knapp adjustment
    q = np.sum(w * (y - pooled_logit)**2) / (k - 1)

    se_hk = np.sqrt(q / np.sum(w))

    tcrit = t.ppf(0.975, df=k-1)

    lower = pooled_logit - tcrit * se_hk
    upper = pooled_logit + tcrit * se_hk

    # Back transformation
    inv_logit = lambda x: np.exp(x) / (1 + np.exp(x))

    pooled_prop = inv_logit(pooled_logit)
    lower_prop = inv_logit(lower)
    upper_prop = inv_logit(upper)

    # I²
    w_fixed = 1 / v
    mu_fixed = np.sum(w_fixed * y) / np.sum(w_fixed)
    Q = np.sum(w_fixed * (y - mu_fixed)**2)

    I2 = max(0, (Q - (k - 1)) / Q) * 100 if Q > 0 else 0

    return {
        "k": k,
        "pooled_AL": pooled_prop * 100,
        "lower_CI": lower_prop * 100,
        "upper_CI": upper_prop * 100,
        "tau2": tau2,
        "I2": I2
    }


# ------------------------------------------------------------
# Run analysis for each technique
# ------------------------------------------------------------

results = []

for group in [
    "circular-stapled",
    "linear-stapled",
    "hand-sewn-stapled"
]:

    subgroup = tech_data[
        tech_data["anastomotic_technique"] == group
    ]

    result = reml_meta(subgroup)
    result["group"] = group

    results.append(result)


import pandas as pd

technique_results = pd.DataFrame(results)

technique_results = technique_results[
    [
        "group",
        "k",
        "pooled_AL",
        "lower_CI",
        "upper_CI",
        "I2",
        "tau2"
    ]
]

print(technique_results.round(2))

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import f

# ============================================================
# Moderator analysis: Anastomotic technique
# Reference category = circular-stapled
# ============================================================

meta_reg = tech_data.copy()

# Dummy variables
meta_reg["linear"] = (
    meta_reg["anastomotic_technique"] == "linear-stapled"
).astype(int)

meta_reg["hand_sewn"] = (
    meta_reg["anastomotic_technique"] == "hand-sewn-stapled"
).astype(int)

# Outcome
y = meta_reg["logit_p"].to_numpy()

# Sampling variance
v = meta_reg["variance"].to_numpy()

# Design matrix:
# intercept + linear + hand-sewn
X = np.column_stack([
    np.ones(len(meta_reg)),
    meta_reg["linear"].to_numpy(),
    meta_reg["hand_sewn"].to_numpy()
])

k = len(y)
p = X.shape[1]


# ============================================================
# REML estimation of tau²
# ============================================================

def reml_objective(tau2):

    W = np.diag(1 / (v + tau2))

    XtWX = X.T @ W @ X
    XtWy = X.T @ W @ y

    beta = np.linalg.solve(XtWX, XtWy)

    residuals = y - X @ beta

    sign, logdet = np.linalg.slogdet(XtWX)

    return 0.5 * (
        np.sum(np.log(v + tau2))
        + logdet
        + residuals.T @ W @ residuals
    )


opt = minimize_scalar(
    reml_objective,
    bounds=(0, 10),
    method="bounded"
)

tau2_mod = opt.x


# ============================================================
# Fit meta-regression
# ============================================================

W = np.diag(1 / (v + tau2_mod))

XtWX = X.T @ W @ X

beta = np.linalg.solve(
    XtWX,
    X.T @ W @ y
)

residuals = y - X @ beta

df_resid = k - p


# ============================================================
# Hartung-Knapp / Knapp-Hartung adjustment
# ============================================================

q = (
    residuals.T @ W @ residuals
    / df_resid
)

cov_beta = q * np.linalg.inv(XtWX)


# ============================================================
# Omnibus test for moderator
# H0: beta_linear = beta_hand_sewn = 0
# ============================================================

b_mod = beta[1:]

V_mod = cov_beta[1:, 1:]

Qm = b_mod.T @ np.linalg.inv(V_mod) @ b_mod

df_num = 2

F_stat = Qm / df_num

p_mod = 1 - f.cdf(
    F_stat,
    df_num,
    df_resid
)


print("Number of cohorts =", k)
print("Residual df =", df_resid)
print("REML tau² =", round(tau2_mod, 4))

print("\nModerator test for anastomotic technique")
print("F =", round(F_stat, 3))
print("df =", df_num, ",", df_resid)
print("P =", round(p_mod, 4))

print("\nCoefficients:")
print("Intercept (circular) =", round(beta[0], 4))
print("Linear vs circular =", round(beta[1], 4))
print("Hand-sewn-stapled vs circular =", round(beta[2], 4))

In [ ]:
from scipy.stats import t
import numpy as np
import pandas as pd

# ============================================================
# Pairwise comparisons of anastomotic techniques
# Based on the fitted REML meta-regression above
# ============================================================

comparisons = {
    "Linear vs Circular": np.array([0, 1, 0]),
    "Hand-sewn-stapled vs Circular": np.array([0, 0, 1]),
    "Hand-sewn-stapled vs Linear": np.array([0, -1, 1])
}

pairwise_results = []

for name, c in comparisons.items():

    estimate = c @ beta

    se = np.sqrt(
        c @ cov_beta @ c
    )

    t_stat = estimate / se

    p_value = 2 * (
        1 - t.cdf(abs(t_stat), df=df_resid)
    )

    lower = estimate - t.ppf(0.975, df_resid) * se
    upper = estimate + t.ppf(0.975, df_resid) * se

    pairwise_results.append({
        "comparison": name,
        "logit_difference": estimate,
        "SE": se,
        "t": t_stat,
        "lower_CI": lower,
        "upper_CI": upper,
        "p_raw": p_value
    })

pairwise_results = pd.DataFrame(pairwise_results)

# Holm correction for 3 pairwise comparisons
from statsmodels.stats.multitest import multipletests

pairwise_results["p_Holm"] = multipletests(
    pairwise_results["p_raw"],
    method="holm"
)[1]

print(pairwise_results.round(4).to_string(index=False))

In [ ]:
df = pd.read_excel("ICG2.xlsx", header=1)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

print(df["anastomosis_location"].value_counts(dropna=False))

In [ ]:
df["anastomosis_location"].value_counts(dropna=False)

In [ ]:
al_data = df[
    [
        "study_id",
        "author",
        "year",
        "number",
        "al_event",
        "al_rate",
        "anastomosis_location"
    ]
].copy()

# Ensure numeric variables are numeric
al_data["number"] = pd.to_numeric(al_data["number"], errors="coerce")
al_data["al_event"] = pd.to_numeric(al_data["al_event"], errors="coerce")

# Recalculate proportion
al_data["proportion"] = (
    al_data["al_event"] / al_data["number"]
)

# Check
al_data[
    [
        "study_id",
        "anastomosis_location",
        "number",
        "al_event",
        "proportion"
    ]
]

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

# Keep only clearly classified locations
location_data = al_data[
    al_data["anastomosis_location"].isin(
        ["intrathoracic", "cervical"]
    )
].copy()

# Logit transformation
location_data["logit_p"] = np.log(
    location_data["proportion"] /
    (1 - location_data["proportion"])
)

# Within-study variance
location_data["variance"] = (
    1 / location_data["al_event"] +
    1 / (location_data["number"] - location_data["al_event"])
)

# REML function
def reml_meta(y, v):

    def neg_reml(tau2):
        w = 1 / (v + tau2)
        mu = np.sum(w * y) / np.sum(w)

        return 0.5 * (
            np.sum(np.log(v + tau2))
            + np.log(np.sum(w))
            + np.sum(w * (y - mu)**2)
        )

    result = minimize_scalar(
        neg_reml,
        bounds=(0, 10),
        method="bounded"
    )

    tau2 = result.x
    w = 1 / (v + tau2)
    mu = np.sum(w * y) / np.sum(w)
    se = np.sqrt(1 / np.sum(w))

    return mu, se, tau2


def inv_logit(x):
    return 1 / (1 + np.exp(-x))


# Run separately for each location
results = []

for group in ["intrathoracic", "cervical"]:

    d = location_data[
        location_data["anastomosis_location"] == group
    ]

    y = d["logit_p"].values
    v = d["variance"].values

    mu, se, tau2 = reml_meta(y, v)

    lower = mu - 1.96 * se
    upper = mu + 1.96 * se

    # I²
    w_fixed = 1 / v
    mu_fixed = np.sum(w_fixed * y) / np.sum(w_fixed)

    Q = np.sum(
        w_fixed * (y - mu_fixed)**2
    )

    df_q = len(d) - 1

    I2 = max(
        0,
        (Q - df_q) / Q * 100
    ) if Q > 0 else 0

    results.append({
        "group": group,
        "k": len(d),
        "pooled_AL": inv_logit(mu) * 100,
        "lower_CI": inv_logit(lower) * 100,
        "upper_CI": inv_logit(upper) * 100,
        "I2": I2,
        "tau2": tau2
    })


location_results = pd.DataFrame(results)

location_results.round(2)

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import t

# Keep only intrathoracic and cervical
meta_loc = location_data.copy()

# Coding:
# cervical = 0 (reference)
# intrathoracic = 1
meta_loc["location_binary"] = (
    meta_loc["anastomosis_location"] == "intrathoracic"
).astype(int)

y = meta_loc["logit_p"].values
v = meta_loc["variance"].values
x = meta_loc["location_binary"].values

X = np.column_stack([
    np.ones(len(y)),
    x
])

# REML objective for meta-regression
def reml_regression(tau2):

    W = np.diag(1 / (v + tau2))

    XtWX = X.T @ W @ X
    XtWy = X.T @ W @ y

    beta = np.linalg.solve(XtWX, XtWy)

    residuals = y - X @ beta

    sign, logdet_X = np.linalg.slogdet(XtWX)

    return 0.5 * (
        np.sum(np.log(v + tau2))
        + logdet_X
        + residuals.T @ W @ residuals
    )

result = minimize_scalar(
    reml_regression,
    bounds=(0, 10),
    method="bounded"
)

tau2 = result.x

# Final model
W = np.diag(1 / (v + tau2))

XtWX = X.T @ W @ X
beta = np.linalg.solve(
    XtWX,
    X.T @ W @ y
)

residuals = y - X @ beta

# Hartung-Knapp adjustment
df_resid = len(y) - X.shape[1]

q = (
    residuals.T @ W @ residuals
) / df_resid

cov_beta = q * np.linalg.inv(XtWX)

se_beta = np.sqrt(
    np.diag(cov_beta)
)

# Moderator coefficient
coef = beta[1]
se = se_beta[1]

t_stat = coef / se

p_value = 2 * (
    1 - t.cdf(abs(t_stat), df=df_resid)
)

tcrit = t.ppf(
    0.975,
    df=df_resid
)

lower = coef - tcrit * se
upper = coef + tcrit * se

print("Number of cohorts =", len(y))
print("Residual df =", df_resid)
print("REML tau² =", round(tau2, 4))

print("\nIntrathoracic vs cervical coefficient =", round(coef, 4))
print("HK SE =", round(se, 4))
print("t =", round(t_stat, 3))
print(
    "95% CI =",
    round(lower, 4),
    "to",
    round(upper, 4)
)
print(
    "P for moderator =",
    round(p_value, 4)
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Use main meta-analysis dataset
x = al_data["logit_p"]
y = al_data["se"]

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    x,
    y,
    s=55
)

# Pooled REML estimate from main analysis
center = pooled_logit_reml

# Funnel limits
se_range = np.linspace(0, y.max() * 1.1, 200)

ax.plot(
    center - 1.96 * se_range,
    se_range,
    linestyle="--"
)

ax.plot(
    center + 1.96 * se_range,
    se_range,
    linestyle="--"
)

ax.axvline(
    center,
    linestyle="--"
)

ax.invert_yaxis()

ax.set_xlabel("Logit-transformed anastomotic leak rate")
ax.set_ylabel("Standard error")

ax.set_title(
    "Funnel plot of anastomotic leak rate"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# ==========================================
# Prepare data for funnel plot
# ==========================================

funnel_data = al_data.copy()

# Proportion
funnel_data["proportion"] = (
    funnel_data["al_event"] / funnel_data["number"]
)

# Logit transformation
funnel_data["logit_p"] = np.log(
    funnel_data["proportion"] /
    (1 - funnel_data["proportion"])
)

# Within-study variance
funnel_data["variance"] = (
    1 / funnel_data["al_event"] +
    1 / (funnel_data["number"] - funnel_data["al_event"])
)

funnel_data["se"] = np.sqrt(
    funnel_data["variance"]
)

# ==========================================
# REML pooled estimate
# ==========================================

y = funnel_data["logit_p"].values
v = funnel_data["variance"].values

def neg_reml(tau2):
    w = 1 / (v + tau2)
    mu = np.sum(w * y) / np.sum(w)

    return 0.5 * (
        np.sum(np.log(v + tau2))
        + np.log(np.sum(w))
        + np.sum(w * (y - mu)**2)
    )

res = minimize_scalar(
    neg_reml,
    bounds=(0, 10),
    method="bounded"
)

tau2_funnel = res.x

w = 1 / (v + tau2_funnel)

pooled_logit_funnel = (
    np.sum(w * y) / np.sum(w)
)

print("REML tau² =", round(tau2_funnel, 4))
print("Pooled logit =", round(pooled_logit_funnel, 4))


# ==========================================
# Funnel plot
# ==========================================

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    funnel_data["logit_p"],
    funnel_data["se"],
    s=55
)

# Range of SE
se_range = np.linspace(
    0,
    funnel_data["se"].max() * 1.05,
    200
)

# Pooled estimate
ax.axvline(
    pooled_logit_funnel,
    linestyle="--",
    linewidth=1.5
)

# Approximate 95% funnel limits
ax.plot(
    pooled_logit_funnel - 1.96 * se_range,
    se_range,
    linestyle="--",
    linewidth=1
)

ax.plot(
    pooled_logit_funnel + 1.96 * se_range,
    se_range,
    linestyle="--",
    linewidth=1
)

# Standard funnel orientation
ax.invert_yaxis()

ax.set_xlabel(
    "Logit-transformed anastomotic leak rate"
)

ax.set_ylabel(
    "Standard error"
)

ax.set_title(
    "Funnel Plot of Anastomotic Leak Rate"
)

plt.tight_layout()

plt.tight_layout()

# Save high-resolution PNG
plt.savefig(
    "Funnel_plot_AL.png",
    dpi=600,
    bbox_inches="tight"
)

# Save vector PDF
plt.savefig(
    "Funnel_plot_AL.pdf",
    bbox_inches="tight"
)



plt.show()

In [ ]:
import numpy as np
import statsmodels.api as sm

# Effect estimate and SE
yi = funnel_data["logit_p"].values
sei = funnel_data["se"].values

# Standard normal deviate
SND = yi / sei

# Precision
precision = 1 / sei

# Egger regression:
# SND = intercept + beta * precision
X = sm.add_constant(precision)

egger_model = sm.OLS(SND, X).fit()

intercept = egger_model.params[0]
se_intercept = egger_model.bse[0]
p_egger = egger_model.pvalues[0]

ci_low, ci_high = egger_model.conf_int(alpha=0.05)[0]

print("Egger's regression test")
print("------------------------")
print("Number of cohorts =", len(yi))
print("Intercept =", round(intercept, 4))
print("SE =", round(se_intercept, 4))
print(
    "95% CI =",
    round(ci_low, 4),
    "to",
    round(ci_high, 4)
)
print("P =", round(p_egger, 4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

# ==========================================
# LEAVE-ONE-OUT SENSITIVITY ANALYSIS
# Random-effects meta-analysis using REML
# ==========================================

loo_data = al_data[
    ["study_id", "number", "al_event"]
].copy()

# Calculate proportion
loo_data["proportion"] = (
    loo_data["al_event"] / loo_data["number"]
)

# Logit transformation
loo_data["logit_p"] = np.log(
    loo_data["proportion"] /
    (1 - loo_data["proportion"])
)

# Within-study variance
loo_data["variance"] = (
    1 / loo_data["al_event"] +
    1 / (loo_data["number"] - loo_data["al_event"])
)


def inv_logit(x):
    return 1 / (1 + np.exp(-x))


def run_reml(data):

    y = data["logit_p"].values
    v = data["variance"].values

    # REML objective
    def neg_reml(tau2):

        w = 1 / (v + tau2)

        mu = np.sum(w * y) / np.sum(w)

        return 0.5 * (
            np.sum(np.log(v + tau2))
            + np.log(np.sum(w))
            + np.sum(w * (y - mu)**2)
        )

    result = minimize_scalar(
        neg_reml,
        bounds=(0, 10),
        method="bounded"
    )

    tau2 = result.x

    # Random-effects weights
    w = 1 / (v + tau2)

    pooled_logit = (
        np.sum(w * y) / np.sum(w)
    )

    se = np.sqrt(
        1 / np.sum(w)
    )

    lower_logit = pooled_logit - 1.96 * se
    upper_logit = pooled_logit + 1.96 * se

    pooled = inv_logit(pooled_logit)
    lower = inv_logit(lower_logit)
    upper = inv_logit(upper_logit)

    # Heterogeneity
    w_fixed = 1 / v

    fixed_mean = (
        np.sum(w_fixed * y) /
        np.sum(w_fixed)
    )

    Q = np.sum(
        w_fixed *
        (y - fixed_mean)**2
    )

    df_q = len(data) - 1

    if Q > 0:
        I2 = max(
            0,
            (Q - df_q) / Q * 100
        )
    else:
        I2 = 0

    return (
        pooled * 100,
        lower * 100,
        upper * 100,
        tau2,
        I2
    )


# ==========================================
# Leave each cohort out once
# ==========================================

loo_results = []

for i in range(len(loo_data)):

    omitted_study = loo_data.iloc[i]["study_id"]

    temp = loo_data.drop(
        loo_data.index[i]
    ).copy()

    pooled, lower, upper, tau2, I2 = run_reml(temp)

    loo_results.append({
        "omitted_study": omitted_study,
        "pooled_AL": pooled,
        "lower_CI": lower,
        "upper_CI": upper,
        "I2": I2,
        "tau2": tau2
    })


loo_results = pd.DataFrame(loo_results)

# Display
loo_results.round({
    "pooled_AL": 2,
    "lower_CI": 2,
    "upper_CI": 2,
    "I2": 1,
    "tau2": 4
})

In [ ]:
# Check ICG dose data

print("ICG dose values:")
print(al_data["dose"].value_counts(dropna=False))

print("\nDose data by cohort:")
display(
    al_data[
        ["study_id", "dose", "number", "al_event", "proportion"]
    ]
)

In [ ]:
al_data["dose"] = df["dose"]

al_data[
    ["study_id", "dose", "number", "al_event", "proportion"]
]

In [ ]:
al_data["stapler_size"] = df["stapler_size"]

display(
    al_data[
        ["study_id", "anastomotic_technique",
         "stapler_size", "number", "al_event", "proportion"]
    ]
)

In [ ]:
# Add stapler size from the original dataframe
al_data["stapler_size"] = df["stapler_size"]

# Display stapler-size data
display(
    al_data[
        ["study_id", "stapler_size", "number", "al_event", "proportion"]
    ]
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import t

# ---------------------------------------------------------
# Publication-year meta-regression
# Random-effects REML + Hartung-Knapp
# ---------------------------------------------------------

year_data = al_data.copy()

# Add year again from original Excel dataframe if needed
if "year" not in year_data.columns:
    year_data["year"] = df["year"]

# Make sure variables are numeric
year_data["year"] = pd.to_numeric(year_data["year"], errors="coerce")
year_data["number"] = pd.to_numeric(year_data["number"], errors="coerce")
year_data["al_event"] = pd.to_numeric(year_data["al_event"], errors="coerce")

# Keep usable cohorts
year_data = year_data.dropna(
    subset=["year", "number", "al_event"]
).copy()

# Calculate proportion
year_data["proportion"] = (
    year_data["al_event"] / year_data["number"]
)

# Logit effect size
year_data["logit_p"] = np.log(
    year_data["proportion"] /
    (1 - year_data["proportion"])
)

# Sampling variance of logit proportion
year_data["variance"] = (
    1 / year_data["al_event"] +
    1 / (year_data["number"] - year_data["al_event"])
)

# Center publication year
year_mean = year_data["year"].mean()
year_data["year_centered"] = year_data["year"] - year_mean

yi = year_data["logit_p"].to_numpy(dtype=float)
vi = year_data["variance"].to_numpy(dtype=float)

X = np.column_stack([
    np.ones(len(year_data)),
    year_data["year_centered"].to_numpy(dtype=float)
])

k = len(yi)
p = X.shape[1]
df_resid = k - p


# ---------------------------------------------------------
# REML objective
# ---------------------------------------------------------

def reml_objective(tau2):

    W = np.diag(1 / (vi + tau2))

    XtWX = X.T @ W @ X
    XtWX_inv = np.linalg.inv(XtWX)

    beta = XtWX_inv @ X.T @ W @ yi

    residuals = yi - X @ beta

    Qe = residuals.T @ W @ residuals

    sign, logdet_X = np.linalg.slogdet(XtWX)

    if sign <= 0:
        return np.inf

    return (
        np.sum(np.log(vi + tau2))
        + logdet_X
        + Qe
    )


result = minimize_scalar(
    reml_objective,
    bounds=(0, 10),
    method="bounded"
)

tau2_reml = result.x


# ---------------------------------------------------------
# Fit final meta-regression
# ---------------------------------------------------------

W = np.diag(1 / (vi + tau2_reml))

XtWX = X.T @ W @ X
XtWX_inv = np.linalg.inv(XtWX)

beta = XtWX_inv @ X.T @ W @ yi

residuals = yi - X @ beta

# Hartung-Knapp scale factor
q = (
    residuals.T @ W @ residuals
) / df_resid

cov_HK = q * XtWX_inv

se_HK = np.sqrt(np.diag(cov_HK))

t_values = beta / se_HK

p_values = 2 * (
    1 - t.cdf(np.abs(t_values), df=df_resid)
)

tcrit = t.ppf(0.975, df=df_resid)

lower = beta - tcrit * se_HK
upper = beta + tcrit * se_HK


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("Publication-year meta-regression")
print("--------------------------------")
print("Number of cohorts =", k)
print("Residual df =", df_resid)
print("Mean publication year =", round(year_mean, 2))
print("REML tau² =", round(tau2_reml, 4))

print("\nEffect of publication year:")
print("Coefficient =", round(beta[1], 4))
print("HK SE =", round(se_HK[1], 4))
print("t =", round(t_values[1], 3))
print(
    "95% CI =",
    round(lower[1], 4),
    "to",
    round(upper[1], 4)
)
print("P for moderator =", round(p_values[1], 4))

# Convert coefficient to odds ratio per 1-year increase
OR = np.exp(beta[1])
OR_low = np.exp(lower[1])
OR_high = np.exp(upper[1])

print("\nOR per 1-year increase =", round(OR, 3))
print(
    "95% CI for OR =",
    round(OR_low, 3),
    "to",
    round(OR_high, 3)
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import t

# =========================================================
# SAMPLE-SIZE META-REGRESSION
# Outcome: logit-transformed AL rate
# Moderator: log(sample size)
# Random-effects REML + Hartung-Knapp
# =========================================================

size_data = al_data.copy()

# Make sure variables are numeric
size_data["number"] = pd.to_numeric(
    size_data["number"], errors="coerce"
)

size_data["al_event"] = pd.to_numeric(
    size_data["al_event"], errors="coerce"
)

# Remove unusable observations
size_data = size_data.dropna(
    subset=["number", "al_event"]
).copy()

size_data = size_data[
    (size_data["number"] > 0) &
    (size_data["al_event"] > 0) &
    (size_data["al_event"] < size_data["number"])
].copy()


# ---------------------------------------------------------
# Effect size and sampling variance
# ---------------------------------------------------------

size_data["proportion"] = (
    size_data["al_event"] /
    size_data["number"]
)

size_data["logit_p"] = np.log(
    size_data["proportion"] /
    (1 - size_data["proportion"])
)

size_data["variance"] = (
    1 / size_data["al_event"] +
    1 / (size_data["number"] - size_data["al_event"])
)


# ---------------------------------------------------------
# Moderator = natural log of sample size
# ---------------------------------------------------------

size_data["log_n"] = np.log(size_data["number"])

mean_log_n = size_data["log_n"].mean()

size_data["log_n_centered"] = (
    size_data["log_n"] - mean_log_n
)


# ---------------------------------------------------------
# Matrices
# ---------------------------------------------------------

yi = size_data["logit_p"].to_numpy(dtype=float)
vi = size_data["variance"].to_numpy(dtype=float)

X = np.column_stack([
    np.ones(len(size_data)),
    size_data["log_n_centered"].to_numpy(dtype=float)
])

k = len(yi)
p = X.shape[1]
df_resid = k - p


# ---------------------------------------------------------
# REML objective
# ---------------------------------------------------------

def reml_objective(tau2):

    W = np.diag(1 / (vi + tau2))

    XtWX = X.T @ W @ X
    XtWX_inv = np.linalg.inv(XtWX)

    beta = XtWX_inv @ X.T @ W @ yi

    residuals = yi - X @ beta

    Qe = residuals.T @ W @ residuals

    sign, logdet_X = np.linalg.slogdet(XtWX)

    if sign <= 0:
        return np.inf

    return (
        np.sum(np.log(vi + tau2))
        + logdet_X
        + Qe
    )


result = minimize_scalar(
    reml_objective,
    bounds=(0, 10),
    method="bounded"
)

tau2_reml = result.x


# ---------------------------------------------------------
# Final random-effects meta-regression
# ---------------------------------------------------------

W = np.diag(1 / (vi + tau2_reml))

XtWX = X.T @ W @ X
XtWX_inv = np.linalg.inv(XtWX)

beta = XtWX_inv @ X.T @ W @ yi

residuals = yi - X @ beta


# Hartung-Knapp adjustment
q = (
    residuals.T @ W @ residuals
) / df_resid

cov_HK = q * XtWX_inv

se_HK = np.sqrt(np.diag(cov_HK))

t_values = beta / se_HK

p_values = 2 * (
    1 - t.cdf(
        np.abs(t_values),
        df=df_resid
    )
)

tcrit = t.ppf(
    0.975,
    df=df_resid
)

lower = beta - tcrit * se_HK
upper = beta + tcrit * se_HK


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("Sample-size meta-regression")
print("--------------------------------")

print("Number of cohorts =", k)
print("Residual df =", df_resid)
print("REML tau² =", round(tau2_reml, 4))

print("\nEffect of log(sample size):")

print(
    "Coefficient =",
    round(beta[1], 4)
)

print(
    "HK SE =",
    round(se_HK[1], 4)
)

print(
    "t =",
    round(t_values[1], 3)
)

print(
    "95% CI =",
    round(lower[1], 4),
    "to",
    round(upper[1], 4)
)

print(
    "P for moderator =",
    round(p_values[1], 4)
)


# ---------------------------------------------------------
# Odds ratio
# ---------------------------------------------------------

OR = np.exp(beta[1])
OR_low = np.exp(lower[1])
OR_high = np.exp(upper[1])

print(
    "\nOR per 1-unit increase in ln(sample size) =",
    round(OR, 3)
)

print(
    "95% CI for OR =",
    round(OR_low, 3),
    "to",
    round(OR_high, 3)
)

In [ ]:
mort30 = df[["study_id", "number", "30-d-mortality"]].copy()

# Convert variables to numeric
# Values such as "n/r" will become NaN
mort30["number"] = pd.to_numeric(
    mort30["number"], errors="coerce"
)

mort30["30-d-mortality"] = pd.to_numeric(
    mort30["30-d-mortality"], errors="coerce"
)

# Exclude studies that did not report 30-day mortality
mort30 = mort30.dropna(
    subset=["number", "30-d-mortality"]
).copy()

mort30["number"] = mort30["number"].astype(int)
mort30["30-d-mortality"] = mort30["30-d-mortality"].astype(int)

# Rename for easier analysis
mort30["events"] = mort30["30-d-mortality"]

print("Number of included cohorts:", len(mort30))

display(
    mort30[
        ["study_id", "number", "events"]
    ]
)

In [ ]:
print(df.columns.tolist())

In [ ]:
mort30 = df[["study_id", "number", "30_d_mortality"]].copy()

# Convert to numeric
# Values such as n/r will become NaN
mort30["number"] = pd.to_numeric(
    mort30["number"], errors="coerce"
)

mort30["30_d_mortality"] = pd.to_numeric(
    mort30["30_d_mortality"], errors="coerce"
)

# Keep only cohorts that actually reported 30-day mortality
mort30 = mort30.dropna(
    subset=["number", "30_d_mortality"]
).copy()

mort30["number"] = mort30["number"].astype(int)
mort30["events"] = mort30["30_d_mortality"].astype(int)

print("Number of included cohorts:", len(mort30))

display(
    mort30[
        ["study_id", "number", "events"]
    ]
)

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import t, chi2

# -----------------------------
# 1. Continuity correction
# -----------------------------

mort30["non_events"] = mort30["number"] - mort30["events"]

mort30["events_cc"] = mort30["events"].astype(float)
mort30["non_events_cc"] = mort30["non_events"].astype(float)

# Apply 0.5 correction only to zero-event studies
zero = mort30["events"] == 0

mort30.loc[zero, "events_cc"] += 0.5
mort30.loc[zero, "non_events_cc"] += 0.5

# Logit transformed proportions
mort30["yi"] = np.log(
    mort30["events_cc"] / mort30["non_events_cc"]
)

# Sampling variance
mort30["vi"] = (
    1 / mort30["events_cc"]
    + 1 / mort30["non_events_cc"]
)

mort30["sei"] = np.sqrt(mort30["vi"])


# -----------------------------
# 2. REML estimation of tau²
# -----------------------------

yi = mort30["yi"].to_numpy()
vi = mort30["vi"].to_numpy()
k = len(yi)

def reml_objective(tau2):

    w = 1 / (vi + tau2)

    mu = np.sum(w * yi) / np.sum(w)

    return (
        np.sum(np.log(vi + tau2))
        + np.log(np.sum(w))
        + np.sum(w * (yi - mu)**2)
    )

upper = max(10.0, np.var(yi, ddof=1) * 10)

res = minimize_scalar(
    reml_objective,
    bounds=(0, upper),
    method="bounded"
)

tau2 = max(0, res.x)

w = 1 / (vi + tau2)

mu = np.sum(w * yi) / np.sum(w)


# -----------------------------
# 3. Hartung-Knapp CI
# -----------------------------

df_hk = k - 1

q_hk = np.sum(
    w * (yi - mu)**2
) / df_hk

se_hk = np.sqrt(
    q_hk / np.sum(w)
)

tcrit = t.ppf(
    0.975,
    df_hk
)

lower_logit = mu - tcrit * se_hk
upper_logit = mu + tcrit * se_hk


def inv_logit(x):
    return 1 / (1 + np.exp(-x))


pooled = inv_logit(mu)
lower = inv_logit(lower_logit)
upper_ci = inv_logit(upper_logit)


# -----------------------------
# 4. Heterogeneity
# -----------------------------

w_fixed = 1 / vi

mu_fixed = (
    np.sum(w_fixed * yi)
    / np.sum(w_fixed)
)

Q = np.sum(
    w_fixed * (yi - mu_fixed)**2
)

df_Q = k - 1

p_Q = 1 - chi2.cdf(
    Q,
    df_Q
)

I2 = (
    max(0, (Q - df_Q) / Q) * 100
    if Q > 0 else 0
)


# -----------------------------
# 5. Results
# -----------------------------

print("30-DAY MORTALITY META-ANALYSIS")
print("--------------------------------")

print(f"Number of cohorts = {k}")

print(
    f"Pooled 30-day mortality = "
    f"{pooled*100:.2f}%"
)

print(
    f"95% HK CI = "
    f"{lower*100:.2f}% to "
    f"{upper_ci*100:.2f}%"
)

print("\nHeterogeneity:")

print(f"Q = {Q:.3f}")
print(f"df = {df_Q}")
print(f"P = {p_Q:.4f}")
print(f"I² = {I2:.1f}%")
print(f"tau² = {tau2:.4f}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.special import expit, roots_hermitenorm
from scipy.optimize import minimize
from scipy.stats import norm

# Raw study-level data
events = mort30["events"].to_numpy(dtype=int)
n = mort30["number"].to_numpy(dtype=int)

# Gauss-Hermite quadrature points
nodes, weights = roots_hermitenorm(40)

# Binomial-normal random-intercept GLMM
# logit(p_i) = beta0 + u_i
# u_i ~ N(0, tau^2)

def neg_loglik(params):
    beta0 = params[0]
    tau = np.exp(params[1])  # guarantees tau > 0

    total_ll = 0.0

    for y, ni in zip(events, n):
        eta = beta0 + tau * nodes
        p = expit(eta)

        # Binomial log-likelihood, excluding the combinatorial
        # constant because it does not depend on model parameters
        log_kernel = (
            y * np.log(p)
            + (ni - y) * np.log1p(-p)
        )

        m = np.max(log_kernel)

        integral = (
            np.sum(weights * np.exp(log_kernel - m))
            / np.sqrt(2 * np.pi)
        )

        total_ll += m + np.log(integral)

    return -total_ll


# Starting value based on crude mortality
crude_p = events.sum() / n.sum()
start_beta = np.log(crude_p / (1 - crude_p))

fit = minimize(
    neg_loglik,
    x0=[start_beta, np.log(0.1)],
    method="Nelder-Mead",
    options={"maxiter": 10000}
)

beta0 = fit.x[0]
tau = np.exp(fit.x[1])
tau2 = tau**2

pooled = expit(beta0)

print("Converged:", fit.success)
print("Number of cohorts:", len(events))
print("Total deaths:", events.sum())
print("Total patients:", n.sum())
print(f"GLMM pooled 30-day mortality = {pooled*100:.2f}%")
print(f"tau = {tau:.4f}")
print(f"tau² = {tau2:.4f}")

In [ ]:
from scipy.optimize import minimize_scalar, brentq
from scipy.stats import chi2

# Minimum negative log-likelihood from fitted GLMM
min_nll = fit.fun

# 95% profile-likelihood cutoff
cutoff = min_nll + chi2.ppf(0.95, 1) / 2


# Profile likelihood for beta0
def profile_nll(beta):

    result = minimize_scalar(
        lambda log_tau: neg_loglik([beta, log_tau]),
        bounds=(-12, 3),
        method="bounded"
    )

    return result.fun


def profile_root(beta):
    return profile_nll(beta) - cutoff


# Find lower and upper limits
lower_beta = brentq(
    profile_root,
    beta0 - 10,
    beta0
)

upper_beta = brentq(
    profile_root,
    beta0,
    beta0 + 10
)


# Convert from logit scale to proportion
lower_glmm = expit(lower_beta)
upper_glmm = expit(upper_beta)


print("GLMM 30-DAY MORTALITY")
print("--------------------------------")
print(f"Number of cohorts = {k}")
print(f"Total deaths = {events.sum()}")
print(f"Total patients = {n.sum()}")
print(f"Pooled mortality = {pooled*100:.2f}%")
print(
    f"95% profile-likelihood CI = "
    f"{lower_glmm*100:.2f}% to {upper_glmm*100:.2f}%"
)
print(f"tau² = {tau2:.4f}")

In [ ]:
# Prepare 90-day mortality data

mort90 = df[["study_id", "number", "90_d_mortality"]].copy()

# Convert to numeric; n/r or other text values become NaN
mort90["number"] = pd.to_numeric(
    mort90["number"], errors="coerce"
)

mort90["90_d_mortality"] = pd.to_numeric(
    mort90["90_d_mortality"], errors="coerce"
)

# Keep only studies that explicitly reported 90-day mortality
mort90 = mort90.dropna(
    subset=["number", "90_d_mortality"]
).copy()

mort90["number"] = mort90["number"].astype(int)
mort90["events"] = mort90["90_d_mortality"].astype(int)

print("Number of included cohorts:", len(mort90))
print("Total patients:", mort90["number"].sum())
print("Total deaths:", mort90["events"].sum())

display(
    mort90[
        ["study_id", "number", "events"]
    ]
)

In [ ]:
import numpy as np
from scipy.special import expit, roots_hermitenorm
from scipy.optimize import minimize

# Raw 90-day mortality data
events90 = mort90["events"].to_numpy(dtype=int)
n90 = mort90["number"].to_numpy(dtype=int)

# Gauss-Hermite quadrature
nodes90, weights90 = roots_hermitenorm(40)

def neg_loglik90(params):
    beta0 = params[0]
    tau = np.exp(params[1])

    total_ll = 0.0

    for y, ni in zip(events90, n90):

        eta = beta0 + tau * nodes90
        p = expit(eta)

        # Prevent numerical log(0)
        p = np.clip(p, 1e-12, 1 - 1e-12)

        log_kernel = (
            y * np.log(p)
            + (ni - y) * np.log1p(-p)
        )

        m = np.max(log_kernel)

        integral = (
            np.sum(
                weights90 * np.exp(log_kernel - m)
            )
            / np.sqrt(2 * np.pi)
        )

        total_ll += m + np.log(integral)

    return -total_ll


# Starting value
crude_p90 = events90.sum() / n90.sum()

start_beta90 = np.log(
    crude_p90 / (1 - crude_p90)
)

fit90 = minimize(
    neg_loglik90,
    x0=[start_beta90, np.log(0.1)],
    method="Nelder-Mead",
    options={"maxiter": 10000}
)

beta90 = fit90.x[0]
tau90 = np.exp(fit90.x[1])
tau2_90 = tau90**2

pooled90 = expit(beta90)

print("Converged:", fit90.success)
print("Number of cohorts:", len(events90))
print("Total deaths:", events90.sum())
print("Total patients:", n90.sum())
print(
    f"GLMM pooled 90-day mortality = "
    f"{pooled90*100:.2f}%"
)
print(f"tau = {tau90:.4f}")
print(f"tau² = {tau2_90:.4f}")

In [ ]:
from scipy.optimize import minimize_scalar, brentq
from scipy.stats import chi2

min_nll90 = fit90.fun

cutoff90 = (
    min_nll90
    + chi2.ppf(0.95, 1) / 2
)

def profile_nll90(beta):

    result = minimize_scalar(
        lambda log_tau:
            neg_loglik90([beta, log_tau]),
        bounds=(-12, 3),
        method="bounded"
    )

    return result.fun


def profile_root90(beta):
    return profile_nll90(beta) - cutoff90


lower_beta90 = brentq(
    profile_root90,
    beta90 - 10,
    beta90
)

upper_beta90 = brentq(
    profile_root90,
    beta90,
    beta90 + 10
)

lower90 = expit(lower_beta90)
upper90 = expit(upper_beta90)

print("\nGLMM 90-DAY MORTALITY")
print("--------------------------------")
print(f"Number of cohorts = {len(events90)}")
print(f"Total deaths = {events90.sum()}")
print(f"Total patients = {n90.sum()}")
print(
    f"Pooled mortality = "
    f"{pooled90*100:.2f}%"
)
print(
    f"95% profile-likelihood CI = "
    f"{lower90*100:.2f}% to "
    f"{upper90*100:.2f}%"
)
print(f"tau² = {tau2_90:.4f}")

In [ ]:
[col for col in df.columns 
 if any(x in str(col).lower() for x in ["pulm", "pneum", "respir"])]

In [ ]:
pulm = df[
    ["study_id", "number", "pulmonary_complications"]
].copy()

# Convert to numeric; n/r or other text becomes NaN
pulm["number"] = pd.to_numeric(
    pulm["number"], errors="coerce"
)

pulm["pulmonary_complications"] = pd.to_numeric(
    pulm["pulmonary_complications"], errors="coerce"
)

# Keep only studies that reported pulmonary complications
pulm = pulm.dropna(
    subset=["number", "pulmonary_complications"]
).copy()

pulm["number"] = pulm["number"].astype(int)
pulm["events"] = pulm["pulmonary_complications"].astype(int)

# Basic summary
print("Number of included cohorts:", len(pulm))
print("Total patients:", pulm["number"].sum())
print("Total pulmonary complications:", pulm["events"].sum())
print("Zero-event cohorts:", (pulm["events"] == 0).sum())

display(
    pulm[
        ["study_id", "number", "events"]
    ]
)

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.stats import t, chi2

# Non-events
pulm["non_events"] = pulm["number"] - pulm["events"]

# Continuity correction only for zero-event cohort
pulm["events_cc"] = pulm["events"].astype(float)
pulm["non_events_cc"] = pulm["non_events"].astype(float)

zero = pulm["events"] == 0

pulm.loc[zero, "events_cc"] += 0.5
pulm.loc[zero, "non_events_cc"] += 0.5

# Logit transformed proportions
pulm["yi"] = np.log(
    pulm["events_cc"] / pulm["non_events_cc"]
)

# Sampling variance
pulm["vi"] = (
    1 / pulm["events_cc"]
    + 1 / pulm["non_events_cc"]
)

yi = pulm["yi"].to_numpy()
vi = pulm["vi"].to_numpy()
k = len(yi)


# -----------------------------
# REML tau²
# -----------------------------

def reml_objective(tau2):

    w = 1 / (vi + tau2)
    mu = np.sum(w * yi) / np.sum(w)

    return (
        np.sum(np.log(vi + tau2))
        + np.log(np.sum(w))
        + np.sum(w * (yi - mu)**2)
    )


upper_bound = max(
    10.0,
    np.var(yi, ddof=1) * 10
)

res = minimize_scalar(
    reml_objective,
    bounds=(0, upper_bound),
    method="bounded"
)

tau2 = max(0, res.x)

w = 1 / (vi + tau2)

mu = np.sum(w * yi) / np.sum(w)


# -----------------------------
# Hartung-Knapp CI
# -----------------------------

df_hk = k - 1

q_hk = (
    np.sum(w * (yi - mu)**2)
    / df_hk
)

se_hk = np.sqrt(
    q_hk / np.sum(w)
)

tcrit = t.ppf(
    0.975,
    df_hk
)

lower_logit = mu - tcrit * se_hk
upper_logit = mu + tcrit * se_hk


def inv_logit(x):
    return 1 / (1 + np.exp(-x))


pooled = inv_logit(mu)
lower = inv_logit(lower_logit)
upper_ci = inv_logit(upper_logit)


# -----------------------------
# Heterogeneity
# -----------------------------

w_fixed = 1 / vi

mu_fixed = (
    np.sum(w_fixed * yi)
    / np.sum(w_fixed)
)

Q = np.sum(
    w_fixed * (yi - mu_fixed)**2
)

df_Q = k - 1

p_Q = 1 - chi2.cdf(Q, df_Q)

I2 = (
    max(0, (Q - df_Q) / Q) * 100
    if Q > 0 else 0
)


# -----------------------------
# Output
# -----------------------------

print("PULMONARY COMPLICATIONS META-ANALYSIS")
print("-------------------------------------")

print(f"Number of cohorts = {k}")
print(f"Total patients = {pulm['number'].sum()}")
print(f"Total events = {pulm['events'].sum()}")

print(
    f"Pooled pulmonary complication rate = "
    f"{pooled*100:.2f}%"
)

print(
    f"95% HK CI = "
    f"{lower*100:.2f}% to "
    f"{upper_ci*100:.2f}%"
)

print("\nHeterogeneity:")
print(f"Q = {Q:.3f}")
print(f"df = {df_Q}")
print(f"P = {p_Q:.4f}")
print(f"I² = {I2:.1f}%")
print(f"tau² = {tau2:.4f}")

In [ ]:
[col for col in df.columns
 if any(x in str(col).lower()
        for x in ["blood", "loss", "hospital", "stay", "los"])]

In [ ]:
descriptive_outcomes = df[
    ["study_id", "number", "los", "blood_loss"]
].copy()

display(descriptive_outcomes)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# ============================================================
# ANASTOMOTIC TECHNIQUE — SUBGROUP FOREST PLOT
# ============================================================

# Keep only clearly classifiable techniques
tech = al_data[
    al_data["anastomotic_technique"].isin([
        "circular-stapled",
        "linear-stapled",
        "hand-sewn-stapled"
    ])
].copy()

# Make sure variables are numeric
tech["number"] = pd.to_numeric(tech["number"], errors="coerce")
tech["al_event"] = pd.to_numeric(tech["al_event"], errors="coerce")

tech = tech.dropna(
    subset=["number", "al_event", "anastomotic_technique"]
).copy()

# ------------------------------------------------------------
# Study-level proportions and 95% CIs
# Wilson CI is used for display of individual study proportions
# ------------------------------------------------------------

def wilson_ci(events, n, alpha=0.05):
    p = events / n
    z = norm.ppf(1 - alpha / 2)

    denom = 1 + z**2 / n

    center = (
        p + z**2 / (2*n)
    ) / denom

    half = (
        z / denom *
        np.sqrt(
            p*(1-p)/n +
            z**2/(4*n**2)
        )
    )

    return max(0, center-half), min(1, center+half)


tech["proportion"] = tech["al_event"] / tech["number"]

ci_values = tech.apply(
    lambda x: wilson_ci(
        x["al_event"],
        x["number"]
    ),
    axis=1
)

tech["ci_low"] = [x[0] for x in ci_values]
tech["ci_high"] = [x[1] for x in ci_values]


# ------------------------------------------------------------
# Subgroup pooled results from our REML/HK analyses
# ------------------------------------------------------------

subgroup_results = {
    "circular-stapled": {
        "label": "Circular-stapled",
        "pooled": 0.1129,
        "low": 0.0586,
        "high": 0.2064,
        "I2": 41.68,
        "tau2": 0.14
    },

    "linear-stapled": {
        "label": "Linear-stapled",
        "pooled": 0.0496,
        "low": 0.0286,
        "high": 0.0848,
        "I2": 0.0,
        "tau2": 0.0
    },

    "hand-sewn-stapled": {
        "label": "Hand-sewn–stapled",
        "pooled": 0.2183,
        "low": 0.1179,
        "high": 0.3684,
        "I2": 13.55,
        "tau2": 0.0
    }
}

group_order = [
    "circular-stapled",
    "linear-stapled",
    "hand-sewn-stapled"
]


# ------------------------------------------------------------
# Build plotting rows
# ------------------------------------------------------------

rows = []

for group in group_order:

    result = subgroup_results[group]

    # Subgroup heading
    rows.append({
        "type": "heading",
        "label": result["label"],
        "group": group
    })

    # Individual studies
    subset = tech[
        tech["anastomotic_technique"] == group
    ].copy()

    for _, row in subset.iterrows():

        rows.append({
            "type": "study",
            "label": row["study_id"],
            "events": int(row["al_event"]),
            "n": int(row["number"]),
            "p": row["proportion"],
            "low": row["ci_low"],
            "high": row["ci_high"],
            "group": group
        })

    # Pooled subgroup result
    rows.append({
        "type": "pooled",
        "label": f'Pooled {result["label"]}',
        "p": result["pooled"],
        "low": result["low"],
        "high": result["high"],
        "I2": result["I2"],
        "tau2": result["tau2"],
        "group": group
    })

    # spacer
    rows.append({
        "type": "space"
    })


# Remove final spacer
rows = rows[:-1]

# y positions
nrows = len(rows)

for i, row in enumerate(rows):
    row["y"] = nrows - i


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 10)
)

# Forest plotting area
ax.set_xlim(-0.48, 0.50)
ax.set_ylim(0, nrows + 2.8)

# Reference line at pooled overall AL = 10.02%
ax.axvline(
    0.1002,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)


for row in rows:

    y = row["y"]

    # -------------------------
    # Subgroup heading
    # -------------------------
    if row["type"] == "heading":

        ax.text(
            -0.47,
            y,
            row["label"],
            fontsize=11,
            fontweight="bold",
            va="center"
        )

    # -------------------------
    # Individual study
    # -------------------------
    elif row["type"] == "study":

        p = row["p"]
        low = row["low"]
        high = row["high"]

        # CI
        ax.plot(
            [low, high],
            [y, y],
            linewidth=1.2
        )

        # Point estimate
        ax.scatter(
            p,
            y,
            marker="s",
            s=40,
            zorder=3
        )

        # Study name
        ax.text(
            -0.47,
            y,
            row["label"],
            fontsize=9,
            va="center"
        )

        # Events / total
        ax.text(
            -0.13,
            y,
            f'{row["events"]}/{row["n"]}',
            fontsize=9,
            ha="center",
            va="center"
        )

        # Effect + CI
        ax.text(
            0.49,
            y,
            (
                f'{p*100:.1f}% '
                f'[{low*100:.1f}, '
                f'{high*100:.1f}]'
            ),
            fontsize=9,
            ha="right",
            va="center"
        )

    # -------------------------
    # Pooled subgroup
    # -------------------------
    elif row["type"] == "pooled":

        p = row["p"]
        low = row["low"]
        high = row["high"]

        # Diamond
        diamond_x = [
            low,
            p,
            high,
            p
        ]

        diamond_y = [
            y,
            y + 0.25,
            y,
            y - 0.25
        ]

        ax.fill(
            diamond_x,
            diamond_y,
            alpha=0.8
        )

        ax.text(
            -0.47,
            y,
            row["label"],
            fontsize=9.5,
            fontweight="bold",
            va="center"
        )

        ax.text(
            0.49,
            y,
            (
                f'{p*100:.2f}% '
                f'[{low*100:.2f}, '
                f'{high*100:.2f}]'
            ),
            fontsize=9.5,
            fontweight="bold",
            ha="right",
            va="center"
        )

        # Heterogeneity
        ax.text(
            -0.47,
            y - 0.55,
            (
                f'I² = {row["I2"]:.1f}%; '
                f'τ² = {row["tau2"]:.3f}'
            ),
            fontsize=8,
            va="center"
        )


# ============================================================
# Headers
# ============================================================

header_y = nrows + 1.4

ax.text(
    -0.47,
    header_y,
    "Study",
    fontsize=10,
    fontweight="bold"
)

ax.text(
    -0.13,
    header_y,
    "Events/Total",
    fontsize=10,
    fontweight="bold",
    ha="center"
)

ax.text(
    0.49,
    header_y,
    "AL rate [95% CI]",
    fontsize=10,
    fontweight="bold",
    ha="right"
)


# Moderator test
ax.text(
    -0.47,
    0.45,
    "Test for subgroup differences: "
    "F(2,10) = 9.90, P = 0.0043",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# Axis formatting
# ============================================================

ax.set_xlabel(
    "Anastomotic leak rate",
    fontsize=10
)

ax.set_xticks(
    [0, 0.1, 0.2, 0.3, 0.4]
)

ax.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%"]
)

ax.set_yticks([])

# Remove unnecessary borders
ax.spines["left"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.title(
    "Anastomotic Leak Rate According to Anastomotic Technique",
    fontsize=13,
    fontweight="bold",
    pad=20
)

plt.tight_layout()

# Save BEFORE show
plt.savefig(
    "forest_anastomotic_technique.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_technique.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
print([col for col in df.columns if "tech" in col.lower()])

In [ ]:
al_data["anastomotic_technique"] = df["anastomotic_technique"].values

In [ ]:
print(al_data["anastomotic_technique"].value_counts(dropna=False))

display(
    al_data[
        ["study_id", "anastomotic_technique", "number", "al_event"]
    ]
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# ============================================================
# FOREST PLOT — ANASTOMOTIC TECHNIQUE
# ============================================================

# Keep only the 3 techniques included in formal subgroup analysis
groups = [
    "circular-stapled",
    "linear-stapled",
    "hand-sewn-stapled"
]

tech = al_data[
    al_data["anastomotic_technique"].isin(groups)
].copy()

tech["number"] = pd.to_numeric(tech["number"], errors="coerce")
tech["al_event"] = pd.to_numeric(tech["al_event"], errors="coerce")

tech = tech.dropna(
    subset=["number", "al_event", "anastomotic_technique"]
).copy()


# ============================================================
# Wilson 95% CI for individual studies
# ============================================================

def wilson_ci(events, n, alpha=0.05):
    p = events / n
    z = norm.ppf(1 - alpha / 2)

    denominator = 1 + z**2 / n

    centre = (
        p + z**2 / (2*n)
    ) / denominator

    half_width = (
        z / denominator
        * np.sqrt(
            p*(1-p)/n +
            z**2/(4*n**2)
        )
    )

    return (
        max(0, centre-half_width),
        min(1, centre+half_width)
    )


tech["proportion"] = tech["al_event"] / tech["number"]

cis = tech.apply(
    lambda r: wilson_ci(r["al_event"], r["number"]),
    axis=1
)

tech["ci_low"] = [x[0] for x in cis]
tech["ci_high"] = [x[1] for x in cis]


# ============================================================
# Final subgroup estimates from REML + Hartung-Knapp analysis
# ============================================================

results = {
    "circular-stapled": {
        "label": "Circular-stapled",
        "estimate": 0.1129,
        "low": 0.0586,
        "high": 0.2064,
        "I2": 41.7,
        "tau2": 0.140
    },

    "linear-stapled": {
        "label": "Linear-stapled",
        "estimate": 0.0496,
        "low": 0.0286,
        "high": 0.0848,
        "I2": 0.0,
        "tau2": 0.000
    },

    "hand-sewn-stapled": {
        "label": "Hand-sewn–stapled",
        "estimate": 0.2183,
        "low": 0.1179,
        "high": 0.3684,
        "I2": 13.6,
        "tau2": 0.000
    }
}


# ============================================================
# Construct rows
# ============================================================

plot_rows = []

for group in groups:

    res = results[group]

    # Group heading
    plot_rows.append({
        "type": "heading",
        "text": res["label"]
    })

    subset = tech[
        tech["anastomotic_technique"] == group
    ]

    # Study rows
    for _, r in subset.iterrows():

        plot_rows.append({
            "type": "study",
            "study": str(r["study_id"]),
            "events": int(r["al_event"]),
            "n": int(r["number"]),
            "estimate": r["proportion"],
            "low": r["ci_low"],
            "high": r["ci_high"]
        })

    # Subgroup pooled row
    plot_rows.append({
        "type": "pooled",
        "text": f'Pooled {res["label"]}',
        "estimate": res["estimate"],
        "low": res["low"],
        "high": res["high"],
        "I2": res["I2"],
        "tau2": res["tau2"]
    })

    plot_rows.append({"type": "space"})


plot_rows = plot_rows[:-1]

# assign y positions
N = len(plot_rows)

for i, row in enumerate(plot_rows):
    row["y"] = N - i


# ============================================================
# CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(13, 10))

# Leave room on left/right for text columns
ax.set_xlim(-0.52, 0.52)
ax.set_ylim(-0.5, N + 3)


# ============================================================
# Overall pooled AL reference line
# ============================================================

ax.axvline(
    0.1002,
    linestyle="--",
    linewidth=1,
    alpha=0.6
)


# ============================================================
# Draw rows
# ============================================================

for row in plot_rows:

    y = row["y"]

    # ------------------------
    # Heading
    # ------------------------
    if row["type"] == "heading":

        ax.text(
            -0.51,
            y,
            row["text"],
            fontsize=11,
            fontweight="bold",
            va="center"
        )

    # ------------------------
    # Individual study
    # ------------------------
    elif row["type"] == "study":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # Confidence interval
        ax.plot(
            [lo, hi],
            [y, y],
            linewidth=1.2
        )

        # Study estimate
        ax.scatter(
            est,
            y,
            marker="s",
            s=42,
            zorder=3
        )

        # Study name
        ax.text(
            -0.51,
            y,
            row["study"],
            fontsize=9,
            va="center"
        )

        # Events / total
        ax.text(
            -0.145,
            y,
            f'{row["events"]}/{row["n"]}',
            fontsize=9,
            ha="center",
            va="center"
        )

        # Rate + CI
        ax.text(
            0.51,
            y,
            (
                f'{100*est:.1f}% '
                f'[{100*lo:.1f}, {100*hi:.1f}]'
            ),
            fontsize=9,
            ha="right",
            va="center"
        )

    # ------------------------
    # Pooled estimate
    # ------------------------
    elif row["type"] == "pooled":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # Diamond
        ax.fill(
            [lo, est, hi, est],
            [y, y+0.27, y, y-0.27],
            alpha=0.85
        )

        ax.text(
            -0.51,
            y,
            row["text"],
            fontsize=9.5,
            fontweight="bold",
            va="center"
        )

        ax.text(
            0.51,
            y,
            (
                f'{100*est:.2f}% '
                f'[{100*lo:.2f}, {100*hi:.2f}]'
            ),
            fontsize=9.5,
            fontweight="bold",
            ha="right",
            va="center"
        )

        # Heterogeneity below pooled row
        ax.text(
            -0.51,
            y - 0.48,
            (
                f'I² = {row["I2"]:.1f}%; '
                f'τ² = {row["tau2"]:.3f}'
            ),
            fontsize=8,
            va="center"
        )


# ============================================================
# Column headers
# ============================================================

header_y = N + 1.4

ax.text(
    -0.51,
    header_y,
    "Study",
    fontsize=10,
    fontweight="bold"
)

ax.text(
    -0.145,
    header_y,
    "Events/Total",
    fontsize=10,
    fontweight="bold",
    ha="center"
)

ax.text(
    0.51,
    header_y,
    "AL rate [95% CI]",
    fontsize=10,
    fontweight="bold",
    ha="right"
)


# ============================================================
# Test for subgroup differences
# ============================================================

ax.text(
    -0.51,
    -0.05,
    r"Test for subgroup differences: $F_{(2,10)}$ = 9.90; P = 0.0043",
    fontsize=9.5,
    fontweight="bold",
    va="center"
)


# ============================================================
# Axis
# ============================================================

ax.set_xlabel(
    "Anastomotic leak rate",
    fontsize=10
)

ax.set_xticks(
    [0, 0.10, 0.20, 0.30, 0.40]
)

ax.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%"]
)

ax.set_yticks([])

ax.spines["left"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)


ax.set_title(
    "Anastomotic Leak Rate According to Anastomotic Technique",
    fontsize=13,
    fontweight="bold",
    pad=20
)


plt.tight_layout()


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "forest_anastomotic_technique.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_technique.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

# ============================================================
# FOREST PLOT — ANASTOMOTIC TECHNIQUE
# ============================================================

groups = [
    "circular-stapled",
    "linear-stapled",
    "hand-sewn-stapled"
]

tech = al_data[
    al_data["anastomotic_technique"].isin(groups)
].copy()

tech["number"] = pd.to_numeric(tech["number"], errors="coerce")
tech["al_event"] = pd.to_numeric(tech["al_event"], errors="coerce")

tech = tech.dropna(
    subset=["number", "al_event", "anastomotic_technique"]
).copy()


# ============================================================
# Study-level estimates on logit scale
# ============================================================

tech["p"] = tech["al_event"] / tech["number"]

tech["yi"] = np.log(
    tech["al_event"] /
    (tech["number"] - tech["al_event"])
)

tech["vi"] = (
    1 / tech["al_event"] +
    1 / (tech["number"] - tech["al_event"])
)

tech["sei"] = np.sqrt(tech["vi"])

tech["ci_low"] = expit(
    tech["yi"] - 1.96 * tech["sei"]
)

tech["ci_high"] = expit(
    tech["yi"] + 1.96 * tech["sei"]
)


# ============================================================
# Final REML + Hartung-Knapp subgroup results
# ============================================================

results = {

    "circular-stapled": {
        "label": "Circular-stapled",
        "estimate": 0.1129,
        "low": 0.0586,
        "high": 0.2064,
        "I2": 41.7,
        "tau2": 0.140
    },

    "linear-stapled": {
        "label": "Linear-stapled",
        "estimate": 0.0496,
        "low": 0.0286,
        "high": 0.0848,
        "I2": 0.0,
        "tau2": 0.000
    },

    "hand-sewn-stapled": {
        "label": "Hand-sewn–stapled",
        "estimate": 0.2183,
        "low": 0.1179,
        "high": 0.3684,
        "I2": 13.6,
        "tau2": 0.000
    }
}


# ============================================================
# Build rows
# ============================================================

rows = []

for group in groups:

    res = results[group]

    rows.append({
        "type": "heading",
        "text": res["label"]
    })

    subset = tech[
        tech["anastomotic_technique"] == group
    ]

    for _, r in subset.iterrows():

        rows.append({
            "type": "study",
            "study": str(r["study_id"]),
            "events": int(r["al_event"]),
            "n": int(r["number"]),
            "estimate": r["p"],
            "low": r["ci_low"],
            "high": r["ci_high"]
        })

    rows.append({
        "type": "pooled",
        "text": f'Pooled {res["label"]}',
        "estimate": res["estimate"],
        "low": res["low"],
        "high": res["high"],
        "I2": res["I2"],
        "tau2": res["tau2"]
    })

    rows.append({"type": "space"})


rows = rows[:-1]

N = len(rows)

for i, row in enumerate(rows):
    row["y"] = N - i


# ============================================================
# Figure
# ============================================================

fig, ax = plt.subplots(figsize=(12, 11))

ax.set_xlim(-0.50, 0.50)
ax.set_ylim(-0.8, N + 2.5)


# ============================================================
# Draw forest
# ============================================================

for row in rows:

    y = row["y"]

    # --------------------------------------------------------
    # Group heading
    # --------------------------------------------------------

    if row["type"] == "heading":

        ax.text(
            -0.49,
            y,
            row["text"],
            fontsize=10.5,
            fontweight="bold",
            va="center"
        )

    # --------------------------------------------------------
    # Individual study
    # --------------------------------------------------------

    elif row["type"] == "study":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # CI
        ax.plot(
            [lo, hi],
            [y, y],
            color="black",
            linewidth=1.0
        )

        # Point estimate
        ax.scatter(
            est,
            y,
            marker="s",
            s=35,
            color="black",
            zorder=3
        )

        # Study
        ax.text(
            -0.49,
            y,
            row["study"],
            fontsize=8.5,
            va="center"
        )

        # Events / total
        ax.text(
            -0.145,
            y,
            f'{row["events"]}/{row["n"]}',
            fontsize=8.5,
            ha="center",
            va="center"
        )

        # Rate + CI
        ax.text(
            0.49,
            y,
            (
                f'{100*est:.1f}% '
                f'[{100*lo:.1f}, {100*hi:.1f}]'
            ),
            fontsize=8.5,
            ha="right",
            va="center"
        )

    # --------------------------------------------------------
    # Pooled subgroup
    # --------------------------------------------------------

    elif row["type"] == "pooled":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # Diamond
        ax.fill(
            [lo, est, hi, est],
            [y, y+0.24, y, y-0.24],
            color="black",
            alpha=0.8
        )

        ax.text(
            -0.49,
            y,
            row["text"],
            fontsize=9,
            fontweight="bold",
            va="center"
        )

        ax.text(
            0.49,
            y,
            (
                f'{100*est:.2f}% '
                f'[{100*lo:.2f}, {100*hi:.2f}]'
            ),
            fontsize=9,
            fontweight="bold",
            ha="right",
            va="center"
        )

        # subgroup heterogeneity
        ax.text(
            -0.49,
            y - 0.43,
            (
                f'I² = {row["I2"]:.1f}%; '
                f'τ² = {row["tau2"]:.3f}'
            ),
            fontsize=7.5,
            va="center"
        )


# ============================================================
# Headers
# ============================================================

header_y = N + 1.2

ax.text(
    -0.49,
    header_y,
    "Study",
    fontsize=9.5,
    fontweight="bold"
)

ax.text(
    -0.145,
    header_y,
    "Events/Total",
    fontsize=9.5,
    fontweight="bold",
    ha="center"
)

ax.text(
    0.49,
    header_y,
    "AL rate [95% CI]",
    fontsize=9.5,
    fontweight="bold",
    ha="right"
)


# ============================================================
# Moderator test
# ============================================================

ax.text(
    -0.49,
    -0.25,
    r"Test for subgroup differences: $F_{2,10}$ = 9.90; P = 0.0043",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# X axis
# ============================================================

ax.set_xlabel(
    "Anastomotic leak rate",
    fontsize=9.5
)

ax.set_xticks(
    [0, 0.10, 0.20, 0.30, 0.40]
)

ax.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%"]
)

ax.set_yticks([])

ax.spines["left"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)


ax.set_title(
    "Anastomotic Leak Rate According to Anastomotic Technique",
    fontsize=12,
    fontweight="bold",
    pad=15
)


plt.tight_layout()


# ============================================================
# Save
# ============================================================

plt.savefig(
    "forest_anastomotic_technique_final.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_technique_final.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# PUBLICATION-STYLE FOREST PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(11, 9))

# Text columns are outside the actual forest region
ax.set_xlim(-0.36, 0.48)
ax.set_ylim(-1.0, N + 2.2)

# Forest plot itself will visually occupy 0–40%
forest_max = 0.40


for row in rows:

    y = row["y"]

    # ========================================================
    # SUBGROUP HEADING
    # ========================================================

    if row["type"] == "heading":

        ax.text(
            -0.35,
            y,
            row["text"],
            fontsize=10,
            fontweight="bold",
            va="center"
        )

    # ========================================================
    # INDIVIDUAL STUDY
    # ========================================================

    elif row["type"] == "study":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # Truncate CI visually at 40%
        displayed_hi = min(hi, forest_max)

        ax.plot(
            [lo, displayed_hi],
            [y, y],
            color="black",
            linewidth=0.9
        )

        # Arrow if CI extends beyond plotting region
        if hi > forest_max:

            ax.annotate(
                "",
                xy=(forest_max, y),
                xytext=(forest_max - 0.018, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color="black",
                    lw=0.9
                )
            )

        # Point estimate
        ax.scatter(
            est,
            y,
            marker="s",
            s=28,
            color="black",
            zorder=3
        )

        # Study name
        ax.text(
            -0.35,
            y,
            row["study"],
            fontsize=8,
            va="center"
        )

        # Events / total
        ax.text(
            -0.10,
            y,
            f'{row["events"]}/{row["n"]}',
            fontsize=8,
            ha="center",
            va="center"
        )

        # Numerical effect
        ax.text(
            0.47,
            y,
            (
                f'{100*est:.1f}% '
                f'[{100*lo:.1f}, {100*hi:.1f}]'
            ),
            fontsize=8,
            ha="right",
            va="center"
        )

    # ========================================================
    # POOLED SUBGROUP
    # ========================================================

    elif row["type"] == "pooled":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        ax.fill(
            [lo, est, hi, est],
            [y, y + 0.20, y, y - 0.20],
            color="black",
            alpha=0.80
        )

        ax.text(
            -0.35,
            y,
            row["text"],
            fontsize=8.5,
            fontweight="bold",
            va="center"
        )

        ax.text(
            0.47,
            y,
            (
                f'{100*est:.2f}% '
                f'[{100*lo:.2f}, {100*hi:.2f}]'
            ),
            fontsize=8.5,
            fontweight="bold",
            ha="right",
            va="center"
        )

        ax.text(
            -0.35,
            y - 0.36,
            (
                f'I² = {row["I2"]:.1f}%; '
                f'τ² = {row["tau2"]:.3f}'
            ),
            fontsize=7,
            va="center"
        )


# ============================================================
# COLUMN HEADINGS
# ============================================================

header_y = N + 1.1

ax.text(
    -0.35,
    header_y,
    "Study",
    fontsize=9,
    fontweight="bold"
)

ax.text(
    -0.10,
    header_y,
    "Events/Total",
    fontsize=9,
    fontweight="bold",
    ha="center"
)

ax.text(
    0.47,
    header_y,
    "AL rate [95% CI]",
    fontsize=9,
    fontweight="bold",
    ha="right"
)


# ============================================================
# MODERATOR TEST
# ============================================================

ax.text(
    -0.35,
    -0.45,
    r"Test for subgroup differences: $F_{2,10}$ = 9.90; P = 0.0043",
    fontsize=8.5,
    fontweight="bold"
)


# ============================================================
# AXIS
# ============================================================

ax.set_xticks(
    [0, 0.10, 0.20, 0.30, 0.40]
)

ax.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%"],
    fontsize=8
)

ax.set_xlabel(
    "Anastomotic leak rate",
    fontsize=9
)

ax.set_yticks([])

ax.spines["left"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

ax.set_title(
    "Anastomotic Leak Rate According to Anastomotic Technique",
    fontsize=11,
    fontweight="bold",
    pad=12
)

plt.tight_layout()

# SAVE FIRST
plt.savefig(
    "forest_anastomotic_technique_final.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_technique_final.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FINAL PUBLICATION-READY FOREST PLOT
# ANASTOMOTIC TECHNIQUE
# Uses the existing "rows" object
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ------------------------------------------------------------
# Figure with 3 separate panels
# ------------------------------------------------------------

fig = plt.figure(figsize=(12, 10))

gs = GridSpec(
    1, 3,
    width_ratios=[2.6, 3.2, 1.9],
    wspace=0.02
)

ax_left   = fig.add_subplot(gs[0, 0])
ax_forest = fig.add_subplot(gs[0, 1])
ax_right  = fig.add_subplot(gs[0, 2])

# Same vertical coordinates for all panels
ymin = -0.8
ymax = N + 2.0

for ax in [ax_left, ax_forest, ax_right]:
    ax.set_ylim(ymin, ymax)


# ============================================================
# LEFT PANEL
# Study + Events/Total
# ============================================================

for row in rows:

    y = row["y"]

    if row["type"] == "heading":

        ax_left.text(
            0.00, y,
            row["text"],
            fontsize=10.5,
            fontweight="bold",
            va="center"
        )

    elif row["type"] == "study":

        ax_left.text(
            0.00, y,
            row["study"],
            fontsize=8.5,
            va="center"
        )

        ax_left.text(
            0.93, y,
            f'{row["events"]}/{row["n"]}',
            fontsize=8.5,
            ha="center",
            va="center"
        )

    elif row["type"] == "pooled":

        ax_left.text(
            0.00, y,
            row["text"],
            fontsize=9,
            fontweight="bold",
            va="center"
        )

        # Heterogeneity
        ax_left.text(
            0.00,
            y - 0.40,
            f'I² = {row["I2"]:.1f}%; τ² = {row["tau2"]:.3f}',
            fontsize=7.5,
            va="center"
        )


# Column headings
ax_left.text(
    0.00,
    N + 1.05,
    "Study",
    fontsize=9.5,
    fontweight="bold"
)

ax_left.text(
    0.93,
    N + 1.05,
    "Events/Total",
    fontsize=9.5,
    fontweight="bold",
    ha="center"
)

ax_left.set_xlim(-0.02, 1.05)
ax_left.axis("off")


# ============================================================
# CENTRAL FOREST PANEL
# ============================================================

forest_max = 0.45

for row in rows:

    y = row["y"]

    if row["type"] == "study":

        est = row["estimate"]
        lo  = row["low"]
        hi  = row["high"]

        # Keep CI within plotting region
        hi_plot = min(hi, forest_max)

        # Confidence interval
        ax_forest.plot(
            [lo, hi_plot],
            [y, y],
            color="black",
            linewidth=1.0
        )

        # Arrow if upper CI extends beyond axis
        if hi > forest_max:

            ax_forest.annotate(
                "",
                xy=(forest_max, y),
                xytext=(forest_max - 0.025, y),
                arrowprops=dict(
                    arrowstyle="->",
                    color="black",
                    lw=1.0
                )
            )

        # Study point
        ax_forest.scatter(
            est,
            y,
            marker="s",
            s=32,
            color="black",
            zorder=3
        )

    elif row["type"] == "pooled":

        est = row["estimate"]
        lo  = row["low"]
        hi  = row["high"]

        # Diamond
        ax_forest.fill(
            [lo, est, hi, est],
            [y, y + 0.22, y, y - 0.22],
            color="black",
            alpha=0.82
        )


# ============================================================
# Forest axis
# ============================================================

ax_forest.set_xlim(0, forest_max)

ax_forest.set_xticks(
    [0, 0.10, 0.20, 0.30, 0.40]
)

ax_forest.set_xticklabels(
    ["0%", "10%", "20%", "30%", "40%"],
    fontsize=8.5
)

ax_forest.set_xlabel(
    "Anastomotic leak rate",
    fontsize=9.5
)

ax_forest.set_yticks([])

ax_forest.spines["left"].set_visible(False)
ax_forest.spines["right"].set_visible(False)
ax_forest.spines["top"].set_visible(False)


# ============================================================
# RIGHT PANEL
# Effect estimates
# ============================================================

for row in rows:

    y = row["y"]

    if row["type"] == "study":

        ax_right.text(
            1.00,
            y,
            (
                f'{100*row["estimate"]:.1f}% '
                f'[{100*row["low"]:.1f}, '
                f'{100*row["high"]:.1f}]'
            ),
            fontsize=8.5,
            ha="right",
            va="center"
        )

    elif row["type"] == "pooled":

        ax_right.text(
            1.00,
            y,
            (
                f'{100*row["estimate"]:.2f}% '
                f'[{100*row["low"]:.2f}, '
                f'{100*row["high"]:.2f}]'
            ),
            fontsize=9,
            fontweight="bold",
            ha="right",
            va="center"
        )


ax_right.text(
    1.00,
    N + 1.05,
    "AL rate [95% CI]",
    fontsize=9.5,
    fontweight="bold",
    ha="right"
)

ax_right.set_xlim(0, 1.02)
ax_right.axis("off")


# ============================================================
# Test for subgroup differences
# ============================================================

fig.text(
    0.06,
    0.045,
    r"Test for subgroup differences: "
    r"$F_{2,10}$ = 9.90; P = 0.0043",
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# Title
# ============================================================

fig.suptitle(
    "Anastomotic Leak Rate According to Anastomotic Technique",
    fontsize=12,
    fontweight="bold",
    y=0.975
)


# ============================================================
# Layout
# ============================================================

plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.93,
    bottom=0.10
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "forest_anastomotic_technique_FINAL.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_technique_FINAL.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
print([col for col in df.columns if "location" in col.lower()])

In [ ]:
al_data["anastomosis_location"] = df["anastomosis_location"].values

print(
    al_data["anastomosis_location"]
    .value_counts(dropna=False)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.special import expit


# ============================================================
# ANASTOMOSIS LOCATION
# ============================================================

location_groups = [
    "intrathoracic",
    "cervical"
]

loc = al_data[
    al_data["anastomosis_location"].isin(location_groups)
].copy()

loc["number"] = pd.to_numeric(
    loc["number"],
    errors="coerce"
)

loc["al_event"] = pd.to_numeric(
    loc["al_event"],
    errors="coerce"
)

loc = loc.dropna(
    subset=[
        "number",
        "al_event",
        "anastomosis_location"
    ]
).copy()


# ============================================================
# STUDY-LEVEL LOGIT ESTIMATES
# ============================================================

loc["p"] = (
    loc["al_event"] /
    loc["number"]
)

loc["yi"] = np.log(
    loc["al_event"] /
    (loc["number"] - loc["al_event"])
)

loc["vi"] = (
    1 / loc["al_event"] +
    1 / (loc["number"] - loc["al_event"])
)

loc["sei"] = np.sqrt(loc["vi"])

loc["ci_low"] = expit(
    loc["yi"] -
    1.96 * loc["sei"]
)

loc["ci_high"] = expit(
    loc["yi"] +
    1.96 * loc["sei"]
)


# ============================================================
# FINAL SUBGROUP RESULTS
# REML + HARTUNG-KNAPP
# ============================================================

location_results = {

    "intrathoracic": {
        "label": "Intrathoracic",
        "estimate": 0.1079,
        "low": 0.0728,
        "high": 0.1571,
        "I2": 72.9,
        "tau2": 0.397
    },

    "cervical": {
        "label": "Cervical",
        "estimate": 0.0680,
        "low": 0.0258,
        "high": 0.1677,
        "I2": 0.0,
        "tau2": 0.000
    }
}


# ============================================================
# BUILD ROWS
# ============================================================

loc_rows = []

for group in location_groups:

    res = location_results[group]

    # Heading
    loc_rows.append({
        "type": "heading",
        "text": res["label"]
    })

    subset = loc[
        loc["anastomosis_location"] == group
    ]

    # Individual studies
    for _, r in subset.iterrows():

        loc_rows.append({
            "type": "study",
            "study": str(r["study_id"]),
            "events": int(r["al_event"]),
            "n": int(r["number"]),
            "estimate": r["p"],
            "low": r["ci_low"],
            "high": r["ci_high"]
        })

    # Pooled subgroup
    loc_rows.append({
        "type": "pooled",
        "text": f'Pooled {res["label"]}',
        "estimate": res["estimate"],
        "low": res["low"],
        "high": res["high"],
        "I2": res["I2"],
        "tau2": res["tau2"]
    })

    loc_rows.append({
        "type": "space"
    })


loc_rows = loc_rows[:-1]

Nloc = len(loc_rows)

for i, row in enumerate(loc_rows):
    row["y"] = Nloc - i


# ============================================================
# CHECK STUDIES BEFORE PLOTTING
# ============================================================

print(
    loc[
        [
            "study_id",
            "anastomosis_location",
            "al_event",
            "number"
        ]
    ].to_string(index=False)
)

print("\nNumber of cohorts:", len(loc))


# ============================================================
# THREE-PANEL FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12, 10)
)

gs = GridSpec(
    1, 3,
    width_ratios=[
        2.6,
        3.2,
        1.9
    ],
    wspace=0.02
)

ax_left = fig.add_subplot(
    gs[0, 0]
)

ax_forest = fig.add_subplot(
    gs[0, 1]
)

ax_right = fig.add_subplot(
    gs[0, 2]
)


ymin = -0.8
ymax = Nloc + 2.0

for ax in [
    ax_left,
    ax_forest,
    ax_right
]:
    ax.set_ylim(
        ymin,
        ymax
    )


# ============================================================
# LEFT PANEL
# ============================================================

for row in loc_rows:

    y = row["y"]

    if row["type"] == "heading":

        ax_left.text(
            0.00,
            y,
            row["text"],
            fontsize=10.5,
            fontweight="bold",
            va="center"
        )

    elif row["type"] == "study":

        ax_left.text(
            0.00,
            y,
            row["study"],
            fontsize=8.5,
            va="center"
        )

        ax_left.text(
            0.93,
            y,
            f'{row["events"]}/{row["n"]}',
            fontsize=8.5,
            ha="center",
            va="center"
        )

    elif row["type"] == "pooled":

        ax_left.text(
            0.00,
            y,
            row["text"],
            fontsize=9,
            fontweight="bold",
            va="center"
        )

        ax_left.text(
            0.00,
            y - 0.40,
            (
                f'I² = '
                f'{row["I2"]:.1f}%; '
                f'τ² = '
                f'{row["tau2"]:.3f}'
            ),
            fontsize=7.5,
            va="center"
        )


# Headers

ax_left.text(
    0.00,
    Nloc + 1.05,
    "Study",
    fontsize=9.5,
    fontweight="bold"
)

ax_left.text(
    0.93,
    Nloc + 1.05,
    "Events/Total",
    fontsize=9.5,
    fontweight="bold",
    ha="center"
)

ax_left.set_xlim(
    -0.02,
    1.05
)

ax_left.axis("off")


# ============================================================
# FOREST PANEL
# ============================================================

forest_max = 0.45

for row in loc_rows:

    y = row["y"]

    if row["type"] == "study":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        hi_plot = min(
            hi,
            forest_max
        )

        # CI
        ax_forest.plot(
            [lo, hi_plot],
            [y, y],
            color="black",
            linewidth=1.0
        )

        # Arrow for truncated CI
        if hi > forest_max:

            ax_forest.annotate(
                "",
                xy=(
                    forest_max,
                    y
                ),
                xytext=(
                    forest_max - 0.025,
                    y
                ),
                arrowprops=dict(
                    arrowstyle="->",
                    color="black",
                    lw=1.0
                )
            )

        # Point estimate
        ax_forest.scatter(
            est,
            y,
            marker="s",
            s=32,
            color="black",
            zorder=3
        )

    elif row["type"] == "pooled":

        est = row["estimate"]
        lo = row["low"]
        hi = row["high"]

        # Diamond
        ax_forest.fill(
            [
                lo,
                est,
                hi,
                est
            ],
            [
                y,
                y + 0.22,
                y,
                y - 0.22
            ],
            color="black",
            alpha=0.82
        )


# ============================================================
# FOREST AXIS
# ============================================================

ax_forest.set_xlim(
    0,
    forest_max
)

ax_forest.set_xticks(
    [
        0,
        0.10,
        0.20,
        0.30,
        0.40
    ]
)

ax_forest.set_xticklabels(
    [
        "0%",
        "10%",
        "20%",
        "30%",
        "40%"
    ],
    fontsize=8.5
)

ax_forest.set_xlabel(
    "Anastomotic leak rate",
    fontsize=9.5
)

ax_forest.set_yticks([])

ax_forest.spines[
    "left"
].set_visible(False)

ax_forest.spines[
    "right"
].set_visible(False)

ax_forest.spines[
    "top"
].set_visible(False)


# ============================================================
# RIGHT PANEL
# ============================================================

for row in loc_rows:

    y = row["y"]

    if row["type"] == "study":

        ax_right.text(
            1.00,
            y,
            (
                f'{100*row["estimate"]:.1f}% '
                f'[{100*row["low"]:.1f}, '
                f'{100*row["high"]:.1f}]'
            ),
            fontsize=8.5,
            ha="right",
            va="center"
        )

    elif row["type"] == "pooled":

        ax_right.text(
            1.00,
            y,
            (
                f'{100*row["estimate"]:.2f}% '
                f'[{100*row["low"]:.2f}, '
                f'{100*row["high"]:.2f}]'
            ),
            fontsize=9,
            fontweight="bold",
            ha="right",
            va="center"
        )


ax_right.text(
    1.00,
    Nloc + 1.05,
    "AL rate [95% CI]",
    fontsize=9.5,
    fontweight="bold",
    ha="right"
)

ax_right.set_xlim(
    0,
    1.02
)

ax_right.axis("off")


# ============================================================
# META-REGRESSION / MODERATOR TEST
# ============================================================

fig.text(
    0.06,
    0.045,
    (
        "Meta-regression for anastomotic location: "
        "β = 0.524; 95% CI, −0.956 to 2.004; "
        "P = 0.458"
    ),
    fontsize=9,
    fontweight="bold"
)


# ============================================================
# TITLE
# ============================================================

fig.suptitle(
    "Anastomotic Leak Rate According to Anastomotic Location",
    fontsize=12,
    fontweight="bold",
    y=0.975
)


plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.93,
    bottom=0.10
)


# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "forest_anastomotic_location_FINAL.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "forest_anastomotic_location_FINAL.pdf",
    bbox_inches="tight"
)

plt.show()